In [1]:
# 셀 1: 환경변수 로드
from dotenv import load_dotenv
import os

load_dotenv()  # .env 파일 자동 로드

API_KEY = os.getenv("DART_API_KEY")
print("API KEY 로드 확인:", API_KEY[:5] + "..." if API_KEY else "❌ 로드 실패")

API KEY 로드 확인: 7898a...


In [3]:
# 셀 3: 전체 corp_code 목록에서 아모레퍼시픽 찾기
import requests
import zipfile
import io
import pandas as pd

url = "https://opendart.fss.or.kr/api/corpCode.xml"
params = {"crtfc_key": API_KEY}

res = requests.get(url, params=params)

# ZIP 해제 → XML 파싱
z = zipfile.ZipFile(io.BytesIO(res.content))
xml_data = z.read(z.namelist()[0])

df = pd.read_xml(io.BytesIO(xml_data))

# 종목코드 090430으로 필터
result = df[df["stock_code"] == "090430"]
print(result[["corp_code", "corp_name", "stock_code"]])

       corp_code corp_name stock_code
84258     583424    아모레퍼시픽     090430


In [4]:
# 셀 4: 아모레퍼시픽 2025년 사업보고서 접수번호 조회
url = "https://opendart.fss.or.kr/api/list.json"
params = {
    "crtfc_key": API_KEY,
    "corp_code": "00583424",  # corp_code (8자리, 앞에 0 패딩)
    "pblntf_ty": "A",          # 정기공시
    "bgn_de": "20250101",
    "end_de": "20251231",
}

res = requests.get(url, params=params)
data = res.json()

for item in data.get("list", []):
    print(item["rcept_no"], "|", item["report_nm"], "|", item["rcept_dt"])

20251114001075 | 분기보고서 (2025.09) | 20251114
20250814001993 | 반기보고서 (2025.06) | 20250814
20250515001510 | 분기보고서 (2025.03) | 20250515
20250317000429 | 사업보고서 (2024.12) | 20250317


In [5]:
# 셀 5: XBRL 원본파일 다운로드
import zipfile
import io
import os

url = "https://opendart.fss.or.kr/api/fnlttXbrl.xml"
params = {
    "crtfc_key": API_KEY,
    "rcept_no": "20250317000429",
    "reprt_code": "11011",  # 사업보고서
}

res = requests.get(url, params=params)

print("Content-Type:", res.headers.get("Content-Type"))
print("Status code:", res.status_code)

# ZIP이면 압축 해제
if "zip" in res.headers.get("Content-Type", "") or res.content[:2] == b'PK':
    z = zipfile.ZipFile(io.BytesIO(res.content))
    os.makedirs("xbrl_amore", exist_ok=True)
    z.extractall("xbrl_amore")
    print("✅ 압축 해제 완료! 파일 목록:")
    for f in z.namelist():
        print(" -", f)
else:
    print("❌ 에러 응답:", res.text)

Content-Type: application/x-msdownload;charset=UTF-8
Status code: 200
✅ 압축 해제 완료! 파일 목록:
 - entity00583424_2024-12-31.xbrl
 - entity00583424_2024-12-31.xsd
 - entity00583424_2024-12-31_def.xml
 - entity00583424_2024-12-31_cal.xml
 - entity00583424_2024-12-31_pre.xml
 - entity00583424_2024-12-31_lab-ko.xml
 - entity00583424_2024-12-31_lab-en.xml


In [6]:
# 셀 6: XBRL 파싱 → 재무수치 추출
from lxml import etree
import pandas as pd

xbrl_path = "xbrl_amore/entity00583424_2024-12-31.xbrl"

tree = etree.parse(xbrl_path)
root = tree.getroot()

# 모든 네임스페이스 확인
print("네임스페이스 목록:")
for prefix, uri in root.nsmap.items():
    print(f"  {prefix}: {uri}")

# 재무 수치 요소 추출 (contextRef 있는 것들)
records = []
for elem in root.iter():
    if elem.get("contextRef") and elem.text and elem.text.strip():
        records.append({
            "tag": elem.tag.split("}")[-1],  # 네임스페이스 제거
            "value": elem.text.strip(),
            "contextRef": elem.get("contextRef"),
            "decimals": elem.get("decimals", ""),
            "unitRef": elem.get("unitRef", ""),
        })

df = pd.DataFrame(records)
print(f"\n총 {len(df)}개 항목 추출")
print(df.head(20))

네임스페이스 목록:
  xbrli: http://www.xbrl.org/2003/instance
  dart-gcd: http://dart.fss.or.kr/taxonomy/2024-06-30/ifrs/dart-gcd
  dart: http://dart.fss.or.kr/taxonomy/2024-06-30/ifrs/dart
  ifrs-full: http://xbrl.ifrs.org/taxonomy/2021-03-24/ifrs-full
  rol_dart: http://dart.fss.or.kr/role/2024-06-30/ifrs/rol_dart
  rol_dart-gcd: http://dart.fss.or.kr/role/2024-06-30/ifrs/rol_dart-gcd
  dart-gaap-businessid: ci
  entity00583424: http://dart.fss.or.kr/taxonomy/2024-12-31/entity00583424
  info: http://xbrl.iasb.org/info
  iso4217: http://www.xbrl.org/2003/iso4217
  link: http://www.xbrl.org/2003/linkbase
  negated: http://www.xbrl.org/2009/role/negated
  net: http://www.xbrl.org/2009/role/net
  nonnum: http://www.xbrl.org/dtr/type/non-numeric
  num: http://www.xbrl.org/dtr/type/numeric
  xbrldi: http://xbrl.org/2006/xbrldi
  xbrldt: http://xbrl.org/2005/xbrldt
  xl: http://www.xbrl.org/2003/XLink
  xlink: http://www.w3.org/1999/xlink
  xsi: http://www.w3.org/2001/XMLSchema-instance

총 7277개 항목

In [7]:
# 셀 7: 주요 재무항목 필터링
# 연결재무제표 + 당기(CurrentYear) 기준으로 필터

# 1. 연결 당기 context만 추출 (CFY = Current Fiscal Year, Consolidated)
cfy_consolidated = df[
    df["contextRef"].str.contains("CFY", na=False) &
    df["contextRef"].str.contains("Consolidated", na=False) &
    ~df["contextRef"].str.contains("Separate", na=False)
]

# 2. 주요 계정과목 키워드
key_tags = [
    "Revenue",                    # 매출액
    "GrossProfit",                # 매출총이익
    "ProfitLossFromOperatingActivities",  # 영업이익
    "ProfitLoss",                 # 당기순이익
    "Assets",                     # 자산총계
    "Liabilities",                # 부채총계
    "Equity",                     # 자본총계
    "CashAndCashEquivalents",     # 현금
]

print("=== 주요 재무항목 (연결, 당기) ===")
for tag in key_tags:
    matched = cfy_consolidated[cfy_consolidated["tag"] == tag]
    if not matched.empty:
        for _, row in matched.iterrows():
            val = int(row["value"]) / 1e8  # 원 → 억원
            print(f"{tag:50s} {val:>15,.0f} 억원")
    else:
        print(f"{tag:50s}  (없음)")

=== 주요 재무항목 (연결, 당기) ===
Revenue                                             (없음)
GrossProfit                                         (없음)
ProfitLossFromOperatingActivities                   (없음)
ProfitLoss                                          (없음)
Assets                                              (없음)
Liabilities                                         (없음)
Equity                                              (없음)
CashAndCashEquivalents                              (없음)


In [8]:
# 셀 8: contextRef 실제 값 샘플 확인
print("=== contextRef 고유값 샘플 ===")
for ctx in df["contextRef"].unique()[:30]:
    print(ctx)

=== contextRef 고유값 샘플 ===
PFY2023eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember_ifrs-full_CarryingAmountAccumulatedDepreciationAmortisationAndImpairmentAndGrossCarryingAmountAxis_ifrs-full_GrossCarryingAmountMember
PFY2023eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember
PFY2023eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_SeparateMember_ifrs-full_CarryingAmountAccumulatedDepreciationAmortisationAndImpairmentAndGrossCarryingAmountAxis_ifrs-full_AccumulatedImpairmentMember
PFY2023eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_SeparateMember_ifrs-full_CarryingAmountAccumulatedDepreciationAmortisationAndImpairmentAndGrossCarryingAmountAxis_ifrs-full_GrossCarryingAmountMember
CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_SeparateMember_ifrs-full_CarryingAmountAccumulatedDepreciationAmortisationAndImpairmentAndGrossCarryingAmount

In [9]:
# 셀 9: contextRef 패턴 수정 후 재필터링
TARGET_CTX = "CFY2024dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"

cfy_con = df[df["contextRef"] == TARGET_CTX]

print(f"해당 context 항목 수: {len(cfy_con)}")
print()

key_tags = {
    "Revenue": "매출액",
    "GrossProfit": "매출총이익",
    "ProfitLossFromOperatingActivities": "영업이익",
    "ProfitLoss": "당기순이익",
    "Assets": "자산총계",
    "Liabilities": "부채총계",
    "Equity": "자본총계",
    "CashAndCashEquivalents": "현금및현금성자산",
}

print("=== 아모레퍼시픽 2024년 연결재무제표 주요항목 ===")
for tag, kor in key_tags.items():
    matched = cfy_con[cfy_con["tag"] == tag]
    if not matched.empty:
        val = int(matched.iloc[0]["value"]) / 1e8
        print(f"{kor:20s} {val:>12,.0f} 억원")
    else:
        print(f"{kor:20s}  (없음)")

# 없는 항목 있으면 전체 tag 목록도 출력
print("\n=== 해당 context 전체 tag 목록 ===")
print(cfy_con["tag"].tolist())

해당 context 항목 수: 158

=== 아모레퍼시픽 2024년 연결재무제표 주요항목 ===
매출액                        38,851 억원
매출총이익                      27,467 억원
영업이익                  (없음)
당기순이익                       6,016 억원
자산총계                  (없음)
부채총계                  (없음)
자본총계                  (없음)
현금및현금성자산              (없음)

=== 해당 context 전체 tag 목록 ===
['DilutedEarningsLossPerSharePreferredStockOfEarningsPerShareAbstract', 'BasicEarningsLossPerSharePreferredStockOfEarningsPerShareAbstract', 'DilutedEarningsLossPerShare', 'BasicEarningsLossPerShare', 'ServicesReceivedRelatedPartyTransactions', 'RevenueFromRenderingOfServicesRelatedPartyTransactions', 'AssetsAndLiabilitiesClassifiedAsHeldforsaleTextBlock', 'FairValueOfInvestmentPropertyWhenEntityAppliesCostModeOfDisclosureOfFairValueMeasurementOfAssetsLineItemsOfDisclosureOfFairValueMeasurementOfAssetsTableOfItems', 'DisclosureOfStatementThatInformationIsNotKnownOrReasonablyEstimableAndEntitysProgressInAssessingExposureToPillarTwoIncomeTaxes', 'DividendsReceive

In [10]:
# 셀 10: 수정된 주요 재무항목 추출

# 손익계산서 항목 → dFY context (당기)
CTX_IS = "CFY2024dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"

# 재무상태표 항목 → eFY context (기말)
CTX_BS = "CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"

is_df = df[df["contextRef"] == CTX_IS]
bs_df = df[df["contextRef"] == CTX_BS]

# 태그 매핑 (실제 DART XBRL 태그명)
is_tags = {
    "Revenue": "매출액",
    "GrossProfit": "매출총이익",
    "OperatingIncomeLoss": "영업이익",
    "ProfitLossBeforeTax": "법인세차감전순이익",
    "ProfitLoss": "당기순이익",
    "ProfitLossAttributableToOwnersOfParent": "지배주주순이익",
}

bs_tags = {
    "Assets": "자산총계",
    "Liabilities": "부채총계",
    "Equity": "자본총계",
    "CashAndCashEquivalents": "현금및현금성자산",
}

print("=== 아모레퍼시픽 2024년 연결 손익계산서 ===")
for tag, kor in is_tags.items():
    matched = is_df[is_df["tag"] == tag]
    if not matched.empty:
        val = int(matched.iloc[0]["value"]) / 1e8
        print(f"  {kor:20s} {val:>12,.0f} 억원")
    else:
        print(f"  {kor:20s}  (없음)")

print("\n=== 아모레퍼시픽 2024년 연결 재무상태표 ===")
for tag, kor in bs_tags.items():
    matched = bs_df[bs_df["tag"] == tag]
    if not matched.empty:
        val = int(matched.iloc[0]["value"]) / 1e8
        print(f"  {kor:20s} {val:>12,.0f} 억원")
    else:
        print(f"  {kor:20s}  (없음)")

=== 아모레퍼시픽 2024년 연결 손익계산서 ===
  매출액                        38,851 억원
  매출총이익                      27,467 억원
  영업이익                        2,205 억원
  법인세차감전순이익                   6,208 억원
  당기순이익                       6,016 억원
  지배주주순이익                     5,932 억원

=== 아모레퍼시픽 2024년 연결 재무상태표 ===
  자산총계                       67,835 억원
  부채총계                       14,575 억원
  자본총계                       53,260 억원
  현금및현금성자산                    4,515 억원


In [11]:
# 셀 11: Taxonomy 조회 (연결 손익계산서 IS1 기준)
url = "https://opendart.fss.or.kr/api/xbrlTaxonomy.json"
params = {
    "crtfc_key": API_KEY,
    "sj_div": "IS1",  # 별개의 손익계산서(연결)
}

res = requests.get(url, params=params)
tax = res.json()

tax_df = pd.DataFrame(tax.get("list", []))
print(f"총 {len(tax_df)}개 계정항목")
print(tax_df[["account_id", "account_nm", "label_kor", "ifrs_ref"]].head(20))

총 361개 계정항목
                                           account_id  \
0                        ifrs_IncomeStatementAbstract   
1                                        ifrs_Revenue   
2                         ifrs_RevenueFromSaleOfGoods   
3                  dart_RevenueFromSaleOfGoodsProduct   
4              dart_RevenueFromSaleOfGoodsMerchandise   
5              dart_RevenueFromSaleOfGoodsRawMaterial   
6                  dart_RevenueFromSaleOfGoodsFishing   
7                     dart_RevenueFromSaleOfGoodsTire   
8                 ifrs_RevenueFromRenderingOfServices   
9                            dart_RevenueFromServices   
10            dart_RevenueFromRenderingOfServicesGame   
11       dart_RevenueFromRenderingOfServicesEducation   
12  dart_RevenueFromRenderingOfServicesFinanceBusi...   
13             dart_RevenueFromSaleOfGoodsGasRecharge   
14  dart_RevenueFromRenderingOfServicesBroadcastBu...   
15  dart_RevenueFromRenderingOfServicesBroadcastPr...   
16    dart_RevenueF

In [12]:
# 셀 12: BS1도 받아서 전체 매핑 테이블 구축
sj_divs = ["BS1", "IS1", "CF1"]  # 연결 재무상태표, 손익, 현금흐름

tax_all = []
for sj in sj_divs:
    res = requests.get("https://opendart.fss.or.kr/api/xbrlTaxonomy.json",
                       params={"crtfc_key": API_KEY, "sj_div": sj})
    rows = res.json().get("list", [])
    for r in rows:
        r["sj_div"] = sj
    tax_all.extend(rows)

tax_master = pd.DataFrame(tax_all)

# account_id에서 접두사(ifrs_, dart_) 제거 → XBRL tag와 매칭
tax_master["tag"] = tax_master["account_id"].str.replace("^ifrs_|^dart_", "", regex=True)

print(f"전체 Taxonomy 항목: {len(tax_master)}개")

# 주요 항목 확인
keywords = ["영업이익", "매출액", "자산총계", "부채총계", "자본총계", "당기순이익", "현금"]
print("\n=== 주요 계정 매핑 확인 ===")
for kw in keywords:
    matched = tax_master[tax_master["label_kor"].str.contains(kw, na=False)]
    if not matched.empty:
        for _, r in matched.head(2).iterrows():
            print(f"  [{r['sj_div']}] {r['label_kor']:20s} → tag: {r['tag']}")

전체 Taxonomy 항목: 1106개

=== 주요 계정 매핑 확인 ===
  [IS1] 영업이익(손실)             → tag: OperatingIncomeLoss
  [IS1] 계속영업이익(손실)           → tag: ProfitLossFromContinuingOperations
  [IS1] 수익(매출액)              → tag: Revenue
  [IS1] 재화의 판매로 인한 수익(매출액)   → tag: RevenueFromSaleOfGoods
  [BS1] 자산총계                 → tag: Assets
  [BS1] 부채총계                 → tag: Liabilities
  [BS1] 자본과부채총계              → tag: EquityAndLiabilities
  [BS1] 자본총계                 → tag: Equity
  [IS1] 당기순이익(손실)            → tag: ProfitLoss
  [IS1] 당기순이익(손실)의 귀속 [abstract] → tag: ProfitLossAttributableToAbstract
  [BS1] 현금및현금성자산             → tag: CashAndCashEquivalents
  [BS1] 현금                   → tag: Cash


In [13]:
# 셀 13: XBRL 파싱 결과 + Taxonomy 결합 → 최종 재무제표

# 1. tag → label_kor 매핑 딕셔너리
tag_to_kor = dict(zip(tax_master["tag"], tax_master["label_kor"]))
tag_to_sj  = dict(zip(tax_master["tag"], tax_master["sj_div"]))

# 2. context 설정
CTX_IS = "CFY2024dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"
CTX_BS = "CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"

# 3. 관심 태그 목록 (Taxonomy에서 확인된 정확한 태그명)
target_tags = {
    # 손익계산서
    "Revenue":                          CTX_IS,
    "GrossProfit":                       CTX_IS,
    "OperatingIncomeLoss":              CTX_IS,
    "ProfitLossBeforeTax":              CTX_IS,
    "ProfitLoss":                        CTX_IS,
    "ProfitLossAttributableToOwnersOfParent": CTX_IS,
    # 재무상태표
    "Assets":                           CTX_BS,
    "Liabilities":                      CTX_BS,
    "Equity":                           CTX_BS,
    "CashAndCashEquivalents":           CTX_BS,
}

# 4. 추출 및 출력
print("=" * 55)
print("  아모레퍼시픽 2024년 연결재무제표 (단위: 억원)")
print("=" * 55)

prev_sj = None
for tag, ctx in target_tags.items():
    sj = tag_to_sj.get(tag, "")
    kor = tag_to_kor.get(tag, tag)

    # 섹션 구분선
    if sj != prev_sj:
        label = "▶ 손익계산서" if "IS" in sj else "▶ 재무상태표"
        print(f"\n{label}")
        prev_sj = sj

    matched = df[(df["tag"] == tag) & (df["contextRef"] == ctx)]
    if not matched.empty:
        val = int(matched.iloc[0]["value"]) / 1e8
        print(f"  {kor:30s} {val:>12,.0f}")
    else:
        print(f"  {kor:30s} {'(없음)':>12}")

print("=" * 55)
print("* 출처: DART OpenAPI XBRL 원본파일 + Taxonomy")

  아모레퍼시픽 2024년 연결재무제표 (단위: 억원)

▶ 손익계산서
  수익(매출액)                              38,851
  매출총이익                                27,467
  영업이익(손실)                              2,205
  법인세비용차감전순이익(손실)                       6,208
  당기순이익(손실)                             6,016
  지배기업의 소유주에게 귀속되는 당기순이익(손실)            5,932

▶ 재무상태표
  자산총계                                 67,835
  부채총계                                 14,575
  자본총계                                 53,260
  현금및현금성자산                              4,515
* 출처: DART OpenAPI XBRL 원본파일 + Taxonomy


In [14]:
# 셀 14: 필요 계정 Taxonomy 탐색
search_keywords = [
    "감가상각", "무형자산상각", "사용권자산상각",  # D&A
    "매출채권", "재고자산", "매입채무",             # NWC
    "유형자산의 취득", "무형자산의 취득",           # CAPEX
    "단기차입금", "유동성장기부채", "유동사채",
    "유동리스부채", "장기차입금", "사채",
    "비유동리스부채",                              # IBD
]

print("=== 서비스 필요 계정 Taxonomy 탐색 결과 ===\n")
for kw in search_keywords:
    matched = tax_master[tax_master["label_kor"].str.contains(kw, na=False)]
    if not matched.empty:
        for _, r in matched.head(2).iterrows():
            print(f"  [{r['sj_div']}] {r['label_kor']:30s} → {r['tag']}")
    else:
        print(f"  ❌ '{kw}' 매칭 없음")
    print()

=== 서비스 필요 계정 Taxonomy 탐색 결과 ===

  [BS1] 감가상각누계액                        → AccumulatedDepreciationCurrentBiologicalAssetsGross
  [BS1] 감가상각누계액                        → AccumulatedDepreciationBuildingsGross

  [IS1] 무형자산상각비                        → AmortisationExpense

  ❌ '사용권자산상각' 매칭 없음

  [BS1] 매출채권 및 기타유동채권                  → TradeAndOtherCurrentReceivables
  [BS1] 매출채권                           → ShortTermTradeReceivable

  [BS1] 재고자산                           → Inventories
  [IS1] 재고자산처분이익                       → GainsOnDisposalsOfInventory

  [BS1] 매입채무 및 기타유동채무                  → TradeAndOtherCurrentPayables
  [BS1] 단기매입채무                         → ShortTermTradePayables

  [CF1] 유형자산의 취득                       → PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities
  [CF1] 유형자산의 취득의 취소                   → CancelPurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities

  [CF1] 무형자산의 취득                       → PurchaseOfIntangibleAssetsClassifiedAsInvestingA

In [15]:
# 셀 15: 전체 필요 계정 — 키워드 포함 탐색 방식으로 통일

search_map = {
    # D&A (IS1)
    "D&A_감가상각":   ("IS1", "tag", "Depreciation"),
    "D&A_무형상각":   ("IS1", "tag", "Amortisation"),
    "D&A_사용권":    ("IS1", "tag", "RightofuseAsset"),

    # NWC (BS1)
    "NWC_매출채권":   ("BS1", "tag", "TradeReceivable"),
    "NWC_재고자산":   ("BS1", "tag", "Inventor"),
    "NWC_매입채무":   ("BS1", "tag", "TradePayable"),

    # CAPEX (CF1)
    "CAPEX_유형":    ("CF1", "tag", "PurchaseOfPropertyPlantAndEquipment"),

    # IBD 유동 (BS1)
    "IBD_단기차입":   ("BS1", "tag", "ShortTermBorrow"),
    "IBD_유동성장기":  ("BS1", "tag", "CurrentPortionOfLongterm"),
    "IBD_유동사채":   ("BS1", "tag", "CurrentPortionOfBond"),
    "IBD_유동리스":   ("BS1", "tag", "CurrentLease"),

    # IBD 비유동 (BS1)
    "IBD_장기차입":   ("BS1", "tag", "LongTermBorrow"),
    "IBD_비유동사채":  ("BS1", "tag", "Bond"),
    "IBD_비유동리스":  ("BS1", "tag", "NoncurrentLease"),

    # NOA 후보 (BS1) — 키워드 기반
    "NOA_투자부동산":  ("BS1", "label_kor", "투자부동산"),
    "NOA_이연법인세":  ("BS1", "label_kor", "이연법인세"),
    "NOA_장기미수금":  ("BS1", "label_kor", "장기미수"),
    "NOA_확정급여":   ("BS1", "label_kor", "확정급여"),
    "NOA_사외적립":   ("BS1", "label_kor", "사외적립"),
}

print("=== 전체 필요 계정 키워드 탐색 결과 ===\n")
results = {}
for name, (sj, field, keyword) in search_map.items():
    matched = tax_master[
        (tax_master["sj_div"] == sj) &
        (tax_master[field].str.contains(keyword, na=False, case=False))
    ]
    results[name] = matched
    print(f"[{name}] → {len(matched)}개 매칭")
    if not matched.empty:
        print(matched[["sj_div", "label_kor", "tag"]].to_string(index=False))
    print()

=== 전체 필요 계정 키워드 탐색 결과 ===

[D&A_감가상각] → 2개 매칭
sj_div  label_kor                                   tag
   IS1      감가상각비                   DepreciationExpense
   IS1 투자부동산감가상각비 InvestmentPropertyDepreciationExpense

[D&A_무형상각] → 1개 매칭
sj_div label_kor                 tag
   IS1   무형자산상각비 AmortisationExpense

[D&A_사용권] → 0개 매칭

[NWC_매출채권] → 5개 매칭
sj_div label_kor                                                       tag
   BS1      매출채권                                  ShortTermTradeReceivable
   BS1     대손충당금      AllowanceForDoubtfulAcccountShortTermTradeReceivable
   BS1    장기매출채권                             LongTermTradeReceivablesGross
   BS1  현재가치할인차금        PresentValueDiscountsLongTermTradeReceivablesGross
   BS1     대손충당금 AllowanceForDoubtfulAcccountLongTermTradeReceivablesGross

[NWC_재고자산] → 3개 매칭
sj_div label_kor                                            tag
   BS1      재고자산                                    Inventories
   BS1      기타재고                          OtherInvento

In [16]:
# 셀 16: 확정 태그 세트 정의

# ── D&A (IS1) ──────────────────────────────────────────
DA_TAGS = {
    "DepreciationExpense":              "감가상각비",
    "AmortisationExpense":              "무형자산상각비",
    # 사용권자산상각은 IS1 Taxonomy 미등재 → XBRL 실데이터에서 직접 탐색
}

# ── NWC (BS1, eFY context) ─────────────────────────────
NWC_TAGS = {
    "ShortTermTradeReceivable":         "매출채권",
    "Inventories":                      "재고자산",
    "ShortTermTradePayables":           "단기매입채무",
}

# ── CAPEX (CF1, dFY context) ───────────────────────────
CAPEX_TAGS = {
    "PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities": "유형자산의 취득",
}

# ── IBD 유동 (BS1, eFY context) ────────────────────────
IBD_CURRENT_TAGS = {
    "ShorttermBorrowings":                  "단기차입금",
    "CurrentPortionOfLongtermBorrowings":   "유동성장기차입금",
    "CurrentPortionOfBonds":                "유동성사채",
    "CurrentPortionOfConvertibleBonds":     "유동성전환사채",
    "CurrentPortionOfBondWithWarrant":      "유동성신주인수권부사채",
    "CurrentPortionOfExchangeableBond":     "유동성교환사채",
    "CurrentLeaseLiabilities":              "유동리스부채",
}

# ── IBD 비유동 (BS1, eFY context) ──────────────────────
IBD_NONCURRENT_TAGS = {
    "LongTermBorrowingsGross":          "장기차입금",
    "BondsIssued":                      "사채",
    "ConvertibleBonds":                 "전환사채",
    "BondWithWarrant":                  "신주인수권부사채",
    "ExchangeableBonds":                "교환사채",
    "NoncurrentLeaseLiabilities":       "비유동리스부채",
}

# ── NOA 후보 (BS1, eFY context) ────────────────────────
# 설계서 v4 기준: 키워드 매칭 → 업종 override → LLM fallback
DEFINITE_NOA_TAGS = {
    "InvestmentProperty":               "투자부동산",
}

COND_NOA_TAGS = {
    "DeferredTaxAssets":                "이연법인세자산",
    "DeferredTaxLiabilities":           "이연법인세부채",
    "LongTermOtherReceivablesGross":    "장기미수금",
    "PresentValueOfDefinedBenefitObligation": "확정급여채무",
    "FairValueOfPlanAssets":            "사외적립자산",
}

print("✅ 태그 세트 정의 완료")
print(f"  D&A        : {len(DA_TAGS)}개 (+사용권자산상각 별도탐색)")
print(f"  NWC        : {len(NWC_TAGS)}개")
print(f"  CAPEX      : {len(CAPEX_TAGS)}개")
print(f"  IBD 유동   : {len(IBD_CURRENT_TAGS)}개")
print(f"  IBD 비유동 : {len(IBD_NONCURRENT_TAGS)}개")
print(f"  NOA 확실   : {len(DEFINITE_NOA_TAGS)}개")
print(f"  NOA 조건부 : {len(COND_NOA_TAGS)}개")

✅ 태그 세트 정의 완료
  D&A        : 2개 (+사용권자산상각 별도탐색)
  NWC        : 3개
  CAPEX      : 1개
  IBD 유동   : 7개
  IBD 비유동 : 6개
  NOA 확실   : 1개
  NOA 조건부 : 5개


In [17]:
# 셀 17: 사용권자산상각 — IS1 Taxonomy 미등재이므로 XBRL 실데이터에서 직접 탐색
rou_candidates = df[
    (df["contextRef"] == CTX_IS) &
    (df["tag"].str.contains("Rightofuse|RightOfUse|ROU", na=False, case=False))
]
print("=== 사용권자산상각 후보 (실데이터) ===")
print(rou_candidates[["tag", "value"]].to_string(index=False))


=== 사용권자산상각 후보 (실데이터) ===
                                                                                                                                                                                                                                                                   tag         value
                                                                                                                                                                 IncreaseDecreaseThroughEffectOfChangesInForeignExchangeRatesLiabilitiesArisingFromFinancingActivities   39770000000
      IncreaseDecreaseThroughFinancingCashFlowsLiabilitiesArisingFromPrincipalOfFinancingActivitiesOfDisclosureOfReconciliationOfLiabilitiesArisingFromFinancingActivitiesLineItemsOfDisclosureOfReconciliationOfLiabilitiesArisingFromFinancingActivitiesTableOfItems  -31051000000
IncreaseDecreaseThroughFinancingCashFlowsLiabilitiesArisingFromInterestExpenseOfFinancingActivitiesOfDisclosureOfReconciliationOfLiabilitiesAri

In [18]:
# 셀 18: context 정의 + 전체 추출 함수

CTX_IS = "CFY2024dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"
CTX_BS = "CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"
CTX_CF = CTX_IS  # 현금흐름표도 dFY 동일

def extract_tags(tag_dict, ctx, unit=1e8, label=""):
    print(f"\n{'='*50}")
    print(f"  {label} (단위: 억원)")
    print(f"{'='*50}")
    total = 0
    found = {}
    for tag, kor in tag_dict.items():
        matched = df[(df["tag"] == tag) & (df["contextRef"] == ctx)]
        if not matched.empty:
            val = int(matched.iloc[0]["value"]) / unit
            found[tag] = val
            print(f"  {kor:30s} {val:>12,.0f}")
            total += val
        else:
            print(f"  {kor:30s} {'(없음)':>12}")
    return found, total

# 실행
da_found,      da_total   = extract_tags(DA_TAGS,           CTX_IS, label="D&A (손익계산서)")
nwc_found,     _          = extract_tags(NWC_TAGS,          CTX_BS, label="NWC 구성 (재무상태표)")
capex_found,   _          = extract_tags(CAPEX_TAGS,        CTX_CF, label="CAPEX (현금흐름표)")
ibd_c_found,   ibd_c_tot  = extract_tags(IBD_CURRENT_TAGS,  CTX_BS, label="IBD 유동 (재무상태표)")
ibd_nc_found,  ibd_nc_tot = extract_tags(IBD_NONCURRENT_TAGS,CTX_BS,label="IBD 비유동 (재무상태표)")
noa_d_found,   _          = extract_tags(DEFINITE_NOA_TAGS, CTX_BS, label="NOA 확실 항목")
noa_c_found,   _          = extract_tags(COND_NOA_TAGS,     CTX_BS, label="NOA 조건부 항목")

# 계산 요약
print("\n" + "="*50)
print("  계산 요약")
print("="*50)

매출채권 = nwc_found.get("ShortTermTradeReceivable", 0)
재고자산  = nwc_found.get("Inventories", 0)
매입채무  = nwc_found.get("ShortTermTradePayables", 0)
NWC      = 매출채권 + 재고자산 - 매입채무

IBD_total = ibd_c_tot + ibd_nc_tot
DA_total  = da_total  # 사용권자산상각 확인 후 추가

print(f"  NWC  = 매출채권({매출채권:,.0f}) + 재고자산({재고자산:,.0f}) - 매입채무({매입채무:,.0f})")
print(f"       = {NWC:,.0f} 억원")
print(f"  IBD  = 유동({ibd_c_tot:,.0f}) + 비유동({ibd_nc_tot:,.0f})")
print(f"       = {IBD_total:,.0f} 억원")
print(f"  D&A  = {DA_total:,.0f} 억원 (사용권자산상각 셀17 확인 후 가산)")


  D&A (손익계산서) (단위: 억원)
  감가상각비                                  (없음)
  무형자산상각비                                (없음)

  NWC 구성 (재무상태표) (단위: 억원)
  매출채권                                   (없음)
  재고자산                                  4,978
  단기매입채무                                 (없음)

  CAPEX (현금흐름표) (단위: 억원)
  유형자산의 취득                                810

  IBD 유동 (재무상태표) (단위: 억원)
  단기차입금                                 3,062
  유동성장기차입금                               (없음)
  유동성사채                                  (없음)
  유동성전환사채                                (없음)
  유동성신주인수권부사채                            (없음)
  유동성교환사채                                (없음)
  유동리스부채                                  613

  IBD 비유동 (재무상태표) (단위: 억원)
  장기차입금                                  (없음)
  사채                                     (없음)
  전환사채                                   (없음)
  신주인수권부사채                               (없음)
  교환사채                                   (없음)
  비유동리스부채                               

In [19]:
# 셀 19: (없음) 항목들 — context 무관하게 실데이터에서 직접 탐색

missing_tags = [
    # D&A
    "DepreciationExpense",
    "AmortisationExpense",
    "DepreciationRightofuseAssets",
    # NWC
    "ShortTermTradeReceivable",
    "ShortTermTradePayables",
    # IBD
    "CurrentPortionOfLongtermBorrowings",
    "LongTermBorrowingsGross",
    "BondsIssued",
    # 확정급여
    "PresentValueOfDefinedBenefitObligation",
    "FairValueOfPlanAssets",
]

print("=== (없음) 항목 실데이터 context 확인 ===\n")
for tag in missing_tags:
    matched = df[df["tag"] == tag]
    if matched.empty:
        print(f"❌ {tag}: 실데이터에 아예 없음 (해당 계정 미사용)")
    else:
        print(f"✅ {tag}: {len(matched)}개 존재")
        for _, r in matched.iterrows():
            val = int(r["value"]) / 1e8 if r["value"].lstrip("-").isdigit() else r["value"]
            print(f"     context: {r['contextRef'][:80]}...")
            print(f"     value  : {val}")
        print()

=== (없음) 항목 실데이터 context 확인 ===

❌ DepreciationExpense: 실데이터에 아예 없음 (해당 계정 미사용)
❌ AmortisationExpense: 실데이터에 아예 없음 (해당 계정 미사용)
✅ DepreciationRightofuseAssets: 28개 존재
     context: PFY2023dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Se...
     value  : 76.09
     context: PFY2023dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Se...
     value  : 2.85
     context: CFY2024dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Se...
     value  : 84.29
     context: CFY2024dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Se...
     value  : 81.26
     context: CFY2024dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Se...
     value  : 3.03
     context: PFY2023dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Se...
     value  : 9.66
     context: PFY2023dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Se...
     value  : 5.83
     context

In [20]:
# 셀 20: 범용 추출 함수 — 우선순위 폴백 + 합산 방식

# ── 우선순위 태그 그룹 정의 ──────────────────────────────
# 리스트 순서 = 우선순위 (앞이 더 구체적/정확)
# 여러 개 found → 합산 / 상위 1개만 쓸지는 mode로 제어

EXTRACTION_RULES = {

    # ── 손익계산서 (CTX_IS) ──────────────────────────────
    "매출액": {
        "ctx": "IS", "mode": "first",
        "tags": ["Revenue"]
    },
    "영업이익": {
        "ctx": "IS", "mode": "first",
        "tags": ["OperatingIncomeLoss"]
    },
    "당기순이익": {
        "ctx": "IS", "mode": "first",
        "tags": ["ProfitLoss"]
    },

    # D&A — 기능별 분류 기업은 개별 항목 없을 수 있음 → 다중 합산
    "감가상각비(유형)": {
        "ctx": "IS", "mode": "first",
        "tags": [
            "DepreciationExpense",                    # 성격별 분류
            "DepreciationAndAmortisationExpense",     # 통합 표시
        ]
    },
    "무형자산상각비": {
        "ctx": "IS", "mode": "first",
        "tags": [
            "AmortisationExpense",
            "AmortisationIntangibleAssetsOtherThanGoodwill",
        ]
    },
    "사용권자산상각비": {
        "ctx": "IS", "mode": "sum_consolidated",  # 연결 context 합산
        "tags": ["DepreciationRightofuseAssets"]
    },

    # ── 재무상태표 (CTX_BS) ──────────────────────────────
    # NWC
    "매출채권": {
        "ctx": "BS", "mode": "first",
        "tags": [
            "ShortTermTradeReceivable",               # 세분화 기업
            "TradeAndOtherCurrentReceivables",        # 묶음 기업 (아모레 등)
            "TradeReceivables",
        ]
    },
    "재고자산": {
        "ctx": "BS", "mode": "first",
        "tags": ["Inventories"]
    },
    "매입채무": {
        "ctx": "BS", "mode": "first",
        "tags": [
            "ShortTermTradePayables",                 # 세분화 기업
            "TradeAndOtherCurrentPayables",           # 묶음 기업
            "TradePayables",
        ]
    },

    # IBD 유동
    "단기차입금": {
        "ctx": "BS", "mode": "first",
        "tags": ["ShorttermBorrowings", "ShortTermBorrowings"]
    },
    "유동성장기차입금": {
        "ctx": "BS", "mode": "first",
        "tags": ["CurrentPortionOfLongtermBorrowings", "CurrentPortionOfLongTermBorrowings"]
    },
    "유동성사채": {
        "ctx": "BS", "mode": "sum",
        "tags": [
            "CurrentPortionOfBonds",
            "CurrentPortionOfConvertibleBonds",
            "CurrentPortionOfBondWithWarrant",
            "CurrentPortionOfExchangeableBond",
        ]
    },
    "유동리스부채": {
        "ctx": "BS", "mode": "first",
        "tags": ["CurrentLeaseLiabilities"]
    },

    # IBD 비유동
    "장기차입금": {
        "ctx": "BS", "mode": "first",
        "tags": ["LongTermBorrowingsGross", "LongtermBorrowings", "LongTermBorrowings"]
    },
    "비유동사채": {
        "ctx": "BS", "mode": "sum",
        "tags": ["BondsIssued", "ConvertibleBonds", "BondWithWarrant", "ExchangeableBonds"]
    },
    "비유동리스부채": {
        "ctx": "BS", "mode": "first",
        "tags": ["NoncurrentLeaseLiabilities"]
    },

    # NOA
    "투자부동산": {
        "ctx": "BS", "mode": "first",
        "tags": ["InvestmentProperty"]
    },
    "이연법인세자산": {
        "ctx": "BS", "mode": "first",
        "tags": ["DeferredTaxAssets"]
    },
    "이연법인세부채": {
        "ctx": "BS", "mode": "first",
        "tags": ["DeferredTaxLiabilities"]
    },
    "확정급여채무": {
        "ctx": "BS", "mode": "first",
        "tags": [
            "PresentValueOfDefinedBenefitObligation",
            "NetDefinedBenefitLiabilityAsset",
            "DefinedBenefitLiability",
        ]
    },
    "사외적립자산": {
        "ctx": "BS", "mode": "first",
        "tags": ["FairValueOfPlanAssets"]
    },

    # CAPEX (CF)
    "유형자산취득(CAPEX)": {
        "ctx": "CF", "mode": "first",
        "tags": ["PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities"]
    },
}

print("✅ EXTRACTION_RULES 정의 완료:", len(EXTRACTION_RULES), "개 항목")

✅ EXTRACTION_RULES 정의 완료: 22 개 항목


In [21]:
# 셀 21: 범용 추출 실행 함수

def extract_all(df, rules, ctx_is, ctx_bs, ctx_cf):
    """
    mode 설명:
    - first           : 태그 우선순위대로 첫 번째 found 값 사용
    - sum             : 해당 context 내 태그들 합산
    - sum_consolidated: Consolidated + dFY context에서 tag 전체 합산
                        (D&A 사용권처럼 segment 분산된 경우)
    """
    ctx_map = {"IS": ctx_is, "BS": ctx_bs, "CF": ctx_cf}
    results = {}

    for item_name, rule in rules.items():
        ctx  = ctx_map[rule["ctx"]]
        mode = rule["mode"]
        tags = rule["tags"]
        val  = None

        if mode == "first":
            for tag in tags:
                matched = df[(df["tag"] == tag) & (df["contextRef"] == ctx)]
                if not matched.empty:
                    val = int(matched.iloc[0]["value"])
                    break

        elif mode == "sum":
            total = 0
            found_any = False
            for tag in tags:
                matched = df[(df["tag"] == tag) & (df["contextRef"] == ctx)]
                if not matched.empty:
                    total += int(matched.iloc[0]["value"])
                    found_any = True
            if found_any:
                val = total

        elif mode == "sum_consolidated":
            # 연결(Consolidated) + 당기(CFY) + dFY context 전체에서 합산
            matched = df[
                df["tag"].isin(tags) &
                df["contextRef"].str.contains("CFY", na=False) &
                df["contextRef"].str.contains("ConsolidatedMember", na=False) &
                ~df["contextRef"].str.contains("SeparateMember", na=False) &
                df["contextRef"].str.contains("dFY", na=False)
            ]
            if not matched.empty:
                val = matched["value"].astype(int).sum()

        results[item_name] = val

    return results

In [22]:
# 셀 22: 실행 + 계산 요약

raw = extract_all(df, EXTRACTION_RULES, CTX_IS, CTX_BS, CTX_CF)

def to_uk(val):  # 원 → 억원
    return val / 1e8 if val is not None else None

def fmt(val):
    return f"{val:>12,.0f}" if val is not None else f"{'(없음)':>12}"

print("=" * 55)
print("  아모레퍼시픽 2024 연결재무제표 추출 결과 (억원)")
print("=" * 55)

sections = {
    "📌 손익계산서": ["매출액","영업이익","당기순이익",
                    "감가상각비(유형)","무형자산상각비","사용권자산상각비"],
    "📌 NWC":       ["매출채권","재고자산","매입채무"],
    "📌 CAPEX":     ["유형자산취득(CAPEX)"],
    "📌 IBD 유동":  ["단기차입금","유동성장기차입금","유동성사채","유동리스부채"],
    "📌 IBD 비유동":["장기차입금","비유동사채","비유동리스부채"],
    "📌 NOA":       ["투자부동산","이연법인세자산","이연법인세부채",
                    "확정급여채무","사외적립자산"],
}

for section, items in sections.items():
    print(f"\n{section}")
    for item in items:
        print(f"  {item:20s} {fmt(to_uk(raw.get(item)))}")

# ── 계산 요약 ──────────────────────────────────────────
print("\n" + "=" * 55)
print("  계산 요약")
print("=" * 55)

DA   = sum(raw[k] or 0 for k in ["감가상각비(유형)","무형자산상각비","사용권자산상각비"])
NWC  = (raw["매출채권"] or 0) + (raw["재고자산"] or 0) - (raw["매입채무"] or 0)
IBD  = sum(raw[k] or 0 for k in [
           "단기차입금","유동성장기차입금","유동성사채","유동리스부채",
           "장기차입금","비유동사채","비유동리스부채"])
EBIT = raw["영업이익"] or 0

print(f"  EBIT    {to_uk(EBIT):>12,.0f} 억원")
print(f"  D&A     {to_uk(DA):>12,.0f} 억원")
print(f"  EBITDA  {to_uk(EBIT+DA):>12,.0f} 억원")
print(f"  NWC     {to_uk(NWC):>12,.0f} 억원")
print(f"  IBD     {to_uk(IBD):>12,.0f} 억원")
print(f"  CAPEX   {to_uk(raw['유형자산취득(CAPEX)'] or 0):>12,.0f} 억원")

  아모레퍼시픽 2024 연결재무제표 추출 결과 (억원)

📌 손익계산서
  매출액                        38,851
  영업이익                        2,205
  당기순이익                       6,016
  감가상각비(유형)                    (없음)
  무형자산상각비                       631
  사용권자산상각비                    1,727

📌 NWC
  매출채권                         (없음)
  재고자산                        4,978
  매입채무                         (없음)

📌 CAPEX
  유형자산취득(CAPEX)                 810

📌 IBD 유동
  단기차입금                       3,062
  유동성장기차입금                     (없음)
  유동성사채                        (없음)
  유동리스부채                        613

📌 IBD 비유동
  장기차입금                        (없음)
  비유동사채                        (없음)
  비유동리스부채                       728

📌 NOA
  투자부동산                       5,947
  이연법인세자산                       509
  이연법인세부채                     2,358
  확정급여채무                       (없음)
  사외적립자산                       (없음)

  계산 요약
  EBIT           2,205 억원
  D&A            2,358 억원
  EBITDA         4,563 억원
  NWC            4,978 억원
  IBD     

In [24]:
# 셀 23: 불일치 항목 실데이터 직접 탐색

check_tags = [
    # D&A 통합 태그
    "DepreciationAndAmortisationExpense",
    # 매출채권 묶음
    "TradeAndOtherCurrentReceivables",
    "TradeReceivables",
    # 매입채무 묶음
    "TradeAndOtherCurrentPayables",
    "TradePayables",
    # NOA 추가 후보
    "CashAndCashEquivalents",
    "FinancialAssets",
    "InvestmentsAccountedForUsingEquityMethod",  # 관계기업투자
]

print("=== 불일치 항목 context 확인 ===\n")
for tag in check_tags:
    matched = df[df["tag"] == tag]
    if matched.empty:
        print(f"❌ {tag}: 실데이터 없음\n")
    else:
        print(f"✅ {tag}: {len(matched)}개")
        # CFY + Consolidated만 필터해서 보여주기
        con = matched[
            matched["contextRef"].str.contains("CFY", na=False) &
            matched["contextRef"].str.contains("Consolidated", na=False) &
            ~matched["contextRef"].str.contains("Separate", na=False)
        ]
        for _, r in con.iterrows():
            val = int(r["value"]) / 1e8
            print(f"  ctx: ...{r['contextRef'][-50:]}")
            print(f"  val: {val:,.0f} 억원")
        print()

=== 불일치 항목 context 확인 ===

✅ DepreciationAndAmortisationExpense: 4개

❌ TradeAndOtherCurrentReceivables: 실데이터 없음

❌ TradeReceivables: 실데이터 없음

❌ TradeAndOtherCurrentPayables: 실데이터 없음

❌ TradePayables: 실데이터 없음

✅ CashAndCashEquivalents: 14개

✅ FinancialAssets: 29개

❌ InvestmentsAccountedForUsingEquityMethod: 실데이터 없음



In [25]:
# 셀 24: Taxonomy 전체에서 관련 계정 전수 탐색

# BS1 전체 로드 (이미 tax_master에 있음)
bs_tax = tax_master[tax_master["sj_div"] == "BS1"]
is_tax = tax_master[tax_master["sj_div"] == "IS1"]
cf_tax = tax_master[tax_master["sj_div"] == "CF1"]

def search_taxonomy(tax_df, keywords, field="label_kor"):
    """키워드 리스트 중 하나라도 포함되면 반환"""
    mask = tax_df[field].str.contains("|".join(keywords), na=False, case=False)
    return tax_df[mask][["sj_div", "label_kor", "tag"]]

# ── 매출채권 계열 ──────────────────────────────────────
print("=== 매출채권 계열 (BS1 Taxonomy 전수) ===")
ar = search_taxonomy(bs_tax, ["매출채권", "receivable", "trade.*receiv"], "label_kor")
ar2 = search_taxonomy(bs_tax, ["TradeReceiv", "Receivable"], "tag")
print(pd.concat([ar, ar2]).drop_duplicates().to_string(index=False))

print("\n=== 매입채무 계열 (BS1 Taxonomy 전수) ===")
ap = search_taxonomy(bs_tax, ["매입채무", "payable", "trade.*pay"], "label_kor")
ap2 = search_taxonomy(bs_tax, ["TradePayable", "Payable"], "tag")
print(pd.concat([ap, ap2]).drop_duplicates().to_string(index=False))

print("\n=== 상각비 계열 (IS1 Taxonomy 전수) ===")
da = search_taxonomy(is_tax, ["감가상각", "상각비", "depreciat", "amortis"], "label_kor")
da2 = search_taxonomy(is_tax, ["Depreciat", "Amortis"], "tag")
print(pd.concat([da, da2]).drop_duplicates().to_string(index=False))

print("\n=== 현금 계열 (BS1 Taxonomy 전수) ===")
cash = search_taxonomy(bs_tax, ["현금", "cash"], "label_kor")
print(cash.to_string(index=False))

print("\n=== 관계기업투자 계열 (BS1 Taxonomy 전수) ===")
eq = search_taxonomy(bs_tax, ["관계기업", "공동기업", "equity.*method", "associate"], "label_kor")
print(eq.to_string(index=False))

=== 매출채권 계열 (BS1 Taxonomy 전수) ===
sj_div        label_kor                                                                    tag
   BS1    매출채권 및 기타유동채권                                        TradeAndOtherCurrentReceivables
   BS1             매출채권                                               ShortTermTradeReceivable
   BS1 장기매출채권 및 기타비유동채권                        LongTermTradeAndOtherNonCurrentReceivablesGross
   BS1           장기매출채권                                          LongTermTradeReceivablesGross
   BS1            대손충당금                   AllowanceForDoubtfulAcccountShortTermTradeReceivable
   BS1         단기금융리스채권                                         CurrentFinanceLeaseReceivables
   BS1            대손충당금             AllowanceForDoubtfulAcccountCurrentFinanceLeaseReceivables
   BS1            단기미수금                                              ShortTermOtherReceivables
   BS1            대손충당금                  AllowanceForDoubtfulAcccountShortTermOtherReceivables
   BS1          

In [26]:
# 셀 25: 실데이터에서 CFY+연결 context 기준으로 매칭 확인
# Taxonomy에 있더라도 실제 기업이 안 쓰면 없으니까 교차검증

def find_in_data(df, tag_list, ctx_is, ctx_bs, ctx_cf):
    """태그 리스트 전체를 실데이터에서 찾아서 context별로 정리"""
    results = []
    for tag in tag_list:
        matched = df[df["tag"] == tag]
        if matched.empty:
            continue
        # CFY + Consolidated만
        con = matched[
            matched["contextRef"].str.contains("CFY", na=False) &
            matched["contextRef"].str.contains("Consolidated", na=False) &
            ~matched["contextRef"].str.contains("Separate", na=False)
        ]
        for _, r in con.iterrows():
            ctx_type = ""
            if "eFY" in r["contextRef"]: ctx_type = "BS(eFY)"
            elif "dFY" in r["contextRef"]: ctx_type = "IS/CF(dFY)"
            results.append({
                "tag": tag,
                "ctx_type": ctx_type,
                "value_억원": int(r["value"]) / 1e8,
                "contextRef_끝50": r["contextRef"][-50:]
            })
    return pd.DataFrame(results)

# 셀24 결과 보고 나서 태그 리스트 채울 예정
# 일단 현재 알려진 후보 전체 투입
candidate_tags = [
    # 매출채권 후보 (Taxonomy 결과 보고 추가)
    "ShortTermTradeReceivable", "TradeAndOtherCurrentReceivables",
    "TradeReceivables", "CurrentTradeReceivables",
    "TradeAndOtherReceivables", "CurrentReceivables",
    # 매입채무 후보
    "ShortTermTradePayables", "TradeAndOtherCurrentPayables",
    "TradePayables", "CurrentTradePayables",
    "TradeAndOtherPayables",
    # D&A 후보
    "DepreciationAndAmortisationExpense",
    "DepreciationExpense", "AmortisationExpense",
    "DepreciationRightofuseAssets",
    # 현금
    "CashAndCashEquivalents", "Cash",
    # 관계기업
    "InvestmentsAccountedForUsingEquityMethod",
    "InvestmentsInSubsidiariesAssociatesAndJointVentures",
]

result_df = find_in_data(df, candidate_tags, CTX_IS, CTX_BS, CTX_CF)
print("=== 실데이터 CFY 연결 기준 매칭 결과 ===")
print(result_df.to_string(index=False))

=== 실데이터 CFY 연결 기준 매칭 결과 ===
Empty DataFrame
Columns: []
Index: []


In [27]:
# 셀 26: 버그 수정 + Taxonomy 결과 반영한 전수 실데이터 탐색

candidate_tags = [
    # 매출채권 계열 (Taxonomy 전수 결과 반영)
    "ShortTermTradeReceivable",
    "TradeAndOtherCurrentReceivables",
    "CurrentTradeReceivables",
    # 매입채무 계열
    "ShortTermTradePayables",
    "TradeAndOtherCurrentPayables",
    "CurrentTradePayables",
    # D&A 계열
    "DepreciationAndAmortisationExpense",
    "DepreciationExpense",
    "AmortisationExpense",
    "DepreciationRightofuseAssets",
    "InvestmentPropertyDepreciationExpense",
    # 현금
    "CashAndCashEquivalents",
    "Cash",
    # 관계기업투자
    "InvestmentsInSubsidiariesJointVenturesAndAssociates",
    "InvestmentsInAssociates",
]

print("=== 실데이터 CFY 연결 기준 전수 탐색 ===\n")
for tag in candidate_tags:
    matched = df[df["tag"] == tag]
    if matched.empty:
        print(f"❌ {tag}")
        continue

    # CFY + Consolidated + Separate 제외
    con = matched[
        matched["contextRef"].str.contains("CFY", na=False) &
        matched["contextRef"].str.contains("Consolidated", na=False) &
        ~matched["contextRef"].str.contains("Separate", na=False)
    ]
    if con.empty:
        print(f"⚠️  {tag}: 실데이터 있으나 CFY+연결 context 없음")
        # 어떤 context로 있는지 샘플 출력
        print(f"    → 실제 context 샘플: {matched.iloc[0]['contextRef'][:80]}")
        continue

    print(f"✅ {tag}")
    for _, r in con.iterrows():
        val = int(r["value"]) / 1e8
        ctx = r["contextRef"]
        ctx_type = "eFY(BS)" if "eFY" in ctx else "dFY(IS/CF)"
        # context에 추가 dimension 있으면 표시
        is_plain = ctx.endswith("ConsolidatedMember")
        flag = "★단순" if is_plain else "  세그먼트"
        print(f"    {flag} [{ctx_type}] {val:>10,.1f}억원")
    print()

=== 실데이터 CFY 연결 기준 전수 탐색 ===

❌ ShortTermTradeReceivable
❌ TradeAndOtherCurrentReceivables
⚠️  CurrentTradeReceivables: 실데이터 있으나 CFY+연결 context 없음
    → 실제 context 샘플: PFY2023eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Se
❌ ShortTermTradePayables
❌ TradeAndOtherCurrentPayables
❌ CurrentTradePayables
⚠️  DepreciationAndAmortisationExpense: 실데이터 있으나 CFY+연결 context 없음
    → 실제 context 샘플: PFY2023dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Co
❌ DepreciationExpense
❌ AmortisationExpense
⚠️  DepreciationRightofuseAssets: 실데이터 있으나 CFY+연결 context 없음
    → 실제 context 샘플: PFY2023dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_Se
❌ InvestmentPropertyDepreciationExpense
⚠️  CashAndCashEquivalents: 실데이터 있으나 CFY+연결 context 없음
    → 실제 context 샘플: BPFY2022eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_C
❌ Cash
⚠️  InvestmentsInSubsidiariesJointVenturesAndAssociates: 실데이터 있으나 CFY+연결 context 없음
    → 실제

In [28]:
# 셀 27: ⚠️ 항목들 context 전체 출력해서 정확히 확인

warning_tags = [
    "CurrentTradeReceivables",
    "DepreciationAndAmortisationExpense", 
    "DepreciationRightofuseAssets",
    "CashAndCashEquivalents",
    "InvestmentsInSubsidiariesJointVenturesAndAssociates",
]

for tag in warning_tags:
    matched = df[df["tag"] == tag]
    # CFY만 필터 (Consolidated/Separate 구분 없이)
    cfy = matched[matched["contextRef"].str.contains("CFY", na=False)]
    print(f"\n{'='*60}")
    print(f"[{tag}] CFY 전체 ({len(cfy)}개)")
    print(f"{'='*60}")
    for _, r in cfy.iterrows():
        val = int(r["value"]) / 1e8
        print(f"  {val:>10,.1f}억  | {r['contextRef']}")


[CurrentTradeReceivables] CFY 전체 (8개)
       -24.9억  | CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_SeparateMember_ifrs-full_CarryingAmountAccumulatedDepreciationAmortisationAndImpairmentAndGrossCarryingAmountAxis_ifrs-full_AccumulatedImpairmentMember
     4,029.6억  | CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_SeparateMember_ifrs-full_CarryingAmountAccumulatedDepreciationAmortisationAndImpairmentAndGrossCarryingAmountAxis_ifrs-full_GrossCarryingAmountMember
     3,863.4억  | CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember
     4,004.7억  | CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_SeparateMember_ifrs-full_CategoriesOfFinancialAssetsAxis_ifrs-full_FinancialAssetsAtAmortisedCostCategoryMember
     3,863.4억  | CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember_ifrs-full_CategoriesOfFinancialAss

In [30]:
# 셀 28: context 추출 함수 완전 재설계 — "단순 context 우선" 원칙

def get_value(df, tag, ctx_type="BS", year="CFY2024"):
    """
    ctx_type: "BS"(eFY), "IS"/"CF"(dFY)
    추출 우선순위:
    1. CFY + ConsolidatedMember 정확히 끝나는 것 (단순 context)
    2. CFY + ConsolidatedMember + ReportedAmountMember (D&A 등)
    3. CFY + ConsolidatedMember 포함 (단순 포함, 첫번째)
    """
    fy = "eFY" if ctx_type == "BS" else "dFY"
    matched = df[df["tag"] == tag]
    if matched.empty:
        return None

    # 우선순위 1: 단순 context (정확히 ConsolidatedMember로 끝남)
    plain_ctx = f"{year}{fy}_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"
    p1 = matched[matched["contextRef"] == plain_ctx]
    if not p1.empty:
        return int(p1.iloc[0]["value"])

    # 우선순위 2: ReportedAmountMember (성격별분류 D&A 등)
    p2 = matched[
        matched["contextRef"].str.contains(f"{year}{fy}", na=False) &
        matched["contextRef"].str.contains("ConsolidatedMember", na=False) &
        ~matched["contextRef"].str.contains("SeparateMember", na=False) &
        matched["contextRef"].str.contains("ReportedAmountMember", na=False)
    ]
    if not p2.empty:
        return int(p2.iloc[0]["value"])

    # 우선순위 3: CFY + Consolidated 포함 (첫번째, 단 Separate 제외)
    p3 = matched[
        matched["contextRef"].str.contains(f"{year}{fy}", na=False) &
        matched["contextRef"].str.contains("ConsolidatedMember", na=False) &
        ~matched["contextRef"].str.contains("SeparateMember", na=False)
    ]
    if not p3.empty:
        return int(p3.iloc[0]["value"])

    return None


# 셀 29: EXTRACTION_RULES 최종 확정 — Taxonomy 전수 결과 반영

EXTRACTION_RULES_FINAL = {

    # ── 손익계산서 ───────────────────────────────────────────
    "매출액": {
        "ctx": "IS",
        "tags": ["Revenue"],
        "mode": "first"
    },
    "영업이익": {
        "ctx": "IS",
        "tags": ["OperatingIncomeLoss"],
        "mode": "first"
    },
    "당기순이익": {
        "ctx": "IS",
        "tags": ["ProfitLoss"],
        "mode": "first"
    },
    "지배주주순이익": {
        "ctx": "IS",
        "tags": ["ProfitLossAttributableToOwnersOfParent"],
        "mode": "first"
    },

    # D&A — 우선순위: 통합표시(성격별) → 개별합산(기능별)
    # 통합태그 있으면 사용, 없으면 개별 3개 합산
    "감가상각및무형상각(통합)": {
        "ctx": "IS",
        "tags": ["DepreciationAndAmortisationExpense"],
        "mode": "reported"   # ReportedAmountMember 우선
    },
    "감가상각비(개별)": {
        "ctx": "IS",
        "tags": ["DepreciationExpense"],
        "mode": "first"
    },
    "무형자산상각비(개별)": {
        "ctx": "IS",
        "tags": ["AmortisationExpense"],
        "mode": "first"
    },
    "사용권자산상각비": {
        "ctx": "IS",
        "tags": ["DepreciationRightofuseAssets"],
        "mode": "first"   # 단순 ConsolidatedMember context
    },

    # ── 재무상태표 ───────────────────────────────────────────
    # NWC
    "매출채권": {
        "ctx": "BS",
        # 세분화기업 → 묶음기업 순서로 fallback
        "tags": [
            "ShortTermTradeReceivable",           # 세분화
            "CurrentTradeReceivables",             # 묶음(아모레 등)
            "TradeAndOtherCurrentReceivables",    # 더 큰 묶음
        ],
        "mode": "first"
    },
    "재고자산": {
        "ctx": "BS",
        "tags": ["Inventories"],
        "mode": "first"
    },
    "매입채무": {
        "ctx": "BS",
        "tags": [
            "ShortTermTradePayables",             # 세분화
            "TradeAndOtherCurrentPayables",       # 묶음
        ],
        "mode": "first"
    },

    # CAPEX
    "유형자산취득(CAPEX)": {
        "ctx": "CF",
        "tags": ["PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities"],
        "mode": "first"
    },

    # IBD 유동
    "단기차입금": {
        "ctx": "BS",
        "tags": ["ShorttermBorrowings", "ShortTermBorrowings"],
        "mode": "first"
    },
    "유동성장기차입금": {
        "ctx": "BS",
        "tags": ["CurrentPortionOfLongtermBorrowings"],
        "mode": "first"
    },
    "유동성사채(합산)": {
        "ctx": "BS",
        "tags": [
            "CurrentPortionOfBonds",
            "CurrentPortionOfConvertibleBonds",
            "CurrentPortionOfBondWithWarrant",
            "CurrentPortionOfExchangeableBond",
        ],
        "mode": "sum"
    },
    "유동리스부채": {
        "ctx": "BS",
        "tags": ["CurrentLeaseLiabilities"],
        "mode": "first"
    },

    # IBD 비유동
    "장기차입금": {
        "ctx": "BS",
        "tags": ["LongTermBorrowingsGross", "LongtermBorrowings"],
        "mode": "first"
    },
    "비유동사채(합산)": {
        "ctx": "BS",
        "tags": [
            "BondsIssued",
            "ConvertibleBonds",
            "BondWithWarrant",
            "ExchangeableBonds",
        ],
        "mode": "sum"
    },
    "비유동리스부채": {
        "ctx": "BS",
        "tags": ["NoncurrentLeaseLiabilities"],
        "mode": "first"
    },

    # NOA
    "현금및현금성자산": {
        "ctx": "BS",
        "tags": ["CashAndCashEquivalents"],
        "mode": "first"
    },
    "투자부동산": {
        "ctx": "BS",
        "tags": ["InvestmentProperty"],
        "mode": "first"
    },
    "이연법인세자산": {
        "ctx": "BS",
        "tags": ["DeferredTaxAssets"],
        "mode": "first"
    },
    "이연법인세부채": {
        "ctx": "BS",
        "tags": ["DeferredTaxLiabilities"],
        "mode": "first"
    },
    "확정급여채무": {
        "ctx": "BS",
        "tags": [
            "PresentValueOfDefinedBenefitObligation",
            "NetDefinedBenefitLiabilityAsset",
            "DefinedBenefitLiability",
        ],
        "mode": "first"
    },
    "사외적립자산": {
        "ctx": "BS",
        "tags": ["FairValueOfPlanAssets"],
        "mode": "first"
    },
}

print("✅ EXTRACTION_RULES_FINAL 정의:", len(EXTRACTION_RULES_FINAL), "개 항목")

✅ EXTRACTION_RULES_FINAL 정의: 25 개 항목


In [32]:
# 셀 29 (누락분): 매입채무/확정급여 실데이터 전수 탐색

ap_tags = [
    "ShortTermTradePayables",
    "TradeAndOtherCurrentPayables",
    "TradePayables",
    "CurrentTradePayables",
    "TradeAndOtherPayables",
]

db_tags = [
    "PresentValueOfDefinedBenefitObligation",
    "NetDefinedBenefitLiabilityAsset",
    "DefinedBenefitLiability",
    "NetDefinedBenefitLiability",
    "EmployeeBenefitsLiability",
]

print("=== 매입채무 계열 CFY 전수 ===\n")
for tag in ap_tags:
    matched = df[df["tag"] == tag]
    if matched.empty:
        print(f"❌ {tag}")
        continue
    cfy = matched[matched["contextRef"].str.contains("CFY", na=False)]
    print(f"✅ {tag}: {len(cfy)}개")
    for _, r in cfy.iterrows():
        val = int(r["value"]) / 1e8
        print(f"   {val:>10,.1f}억 | {r['contextRef'][-70:]}")
    print()

print("\n=== 확정급여 계열 CFY 전수 ===\n")
for tag in db_tags:
    matched = df[df["tag"] == tag]
    if matched.empty:
        print(f"❌ {tag}")
        continue
    cfy = matched[matched["contextRef"].str.contains("CFY", na=False)]
    print(f"✅ {tag}: {len(cfy)}개")
    for _, r in cfy.iterrows():
        val = int(r["value"]) / 1e8
        print(f"   {val:>10,.1f}억 | {r['contextRef'][-70:]}")
    print()

# DART 공시 기준 매입채무 tag 직접 키워드 탐색
print("\n=== BS1 Taxonomy에서 '매입채무' 포함 전수 ===")
ap_all = tax_master[
    (tax_master["sj_div"] == "BS1") &
    (tax_master["label_kor"].str.contains("매입채무", na=False))
]
print(ap_all[["label_kor", "tag"]].to_string(index=False))

=== 매입채무 계열 CFY 전수 ===

❌ ShortTermTradePayables
❌ TradeAndOtherCurrentPayables
❌ TradePayables
❌ CurrentTradePayables
❌ TradeAndOtherPayables

=== 확정급여 계열 CFY 전수 ===

❌ PresentValueOfDefinedBenefitObligation
❌ NetDefinedBenefitLiabilityAsset
❌ DefinedBenefitLiability
❌ NetDefinedBenefitLiability
❌ EmployeeBenefitsLiability

=== BS1 Taxonomy에서 '매입채무' 포함 전수 ===
       label_kor                                     tag
   매입채무 및 기타유동채무            TradeAndOtherCurrentPayables
          단기매입채무                  ShortTermTradePayables
장기매입채무 및 기타비유동채무 LongTermTradeAndOtherNonCurrentPayables
          장기매입채무              LongTermTradePayablesGross


In [31]:
# 셀 30: 최종 추출 실행

def extract_final(df, rules, year="CFY2024"):
    results = {}
    for item_name, rule in rules.items():
        ctx_type = rule["ctx"]
        mode = rule["mode"]
        tags = rule["tags"]

        if mode == "first":
            val = None
            for tag in tags:
                v = get_value(df, tag, ctx_type, year)
                if v is not None:
                    val = v
                    break
            results[item_name] = val

        elif mode == "reported":
            # ReportedAmountMember 우선, 없으면 단순 context
            val = None
            for tag in tags:
                v = get_value(df, tag, ctx_type, year)
                if v is not None:
                    val = v
                    break
            results[item_name] = val

        elif mode == "sum":
            total = 0
            found = False
            for tag in tags:
                v = get_value(df, tag, ctx_type, year)
                if v is not None:
                    total += v
                    found = True
            results[item_name] = total if found else None

    return results

raw = extract_final(df, EXTRACTION_RULES_FINAL)

# ── 출력 ────────────────────────────────────────────────
def fmt(val):
    return f"{val/1e8:>12,.0f}" if val is not None else f"{'(없음)':>12}"

sections = {
    "📌 손익계산서": [
        "매출액","영업이익","당기순이익","지배주주순이익",
        "감가상각및무형상각(통합)","감가상각비(개별)",
        "무형자산상각비(개별)","사용권자산상각비"
    ],
    "📌 NWC":        ["매출채권","재고자산","매입채무"],
    "📌 CAPEX":      ["유형자산취득(CAPEX)"],
    "📌 IBD 유동":   ["단기차입금","유동성장기차입금","유동성사채(합산)","유동리스부채"],
    "📌 IBD 비유동": ["장기차입금","비유동사채(합산)","비유동리스부채"],
    "📌 NOA 후보":   ["현금및현금성자산","투자부동산","이연법인세자산",
                     "이연법인세부채","확정급여채무","사외적립자산"],
}

print("=" * 55)
print("  아모레퍼시픽 2024 연결재무제표 (단위: 억원)")
print("=" * 55)
for section, items in sections.items():
    print(f"\n{section}")
    for item in items:
        print(f"  {item:25s} {fmt(raw.get(item))}")

# ── 계산 요약 ────────────────────────────────────────────
print("\n" + "=" * 55)
print("  계산 요약")
print("=" * 55)

# D&A: 통합 있으면 사용, 없으면 개별 합산
DA_integrated = raw.get("감가상각및무형상각(통합)")
DA_individual = sum(raw.get(k) or 0 for k in
                    ["감가상각비(개별)","무형자산상각비(개별)","사용권자산상각비"])
DA = DA_integrated if DA_integrated else DA_individual
DA_source = "통합태그" if DA_integrated else "개별합산"

EBIT  = raw.get("영업이익") or 0
NWC   = (raw.get("매출채권") or 0) + (raw.get("재고자산") or 0) - (raw.get("매입채무") or 0)
IBD   = sum(raw.get(k) or 0 for k in [
            "단기차입금","유동성장기차입금","유동성사채(합산)","유동리스부채",
            "장기차입금","비유동사채(합산)","비유동리스부채"])
CAPEX = raw.get("유형자산취득(CAPEX)") or 0

print(f"  EBIT    {EBIT/1e8:>10,.0f} 억원")
print(f"  D&A     {DA/1e8:>10,.0f} 억원  ({DA_source})")
print(f"  EBITDA  {(EBIT+DA)/1e8:>10,.0f} 억원")
print(f"  NWC     {NWC/1e8:>10,.0f} 억원")
print(f"  IBD     {IBD/1e8:>10,.0f} 억원")
print(f"  CAPEX   {abs(CAPEX)/1e8:>10,.0f} 억원")

  아모레퍼시픽 2024 연결재무제표 (단위: 억원)

📌 손익계산서
  매출액                             38,851
  영업이익                             2,205
  당기순이익                            6,016
  지배주주순이익                          5,932
  감가상각및무형상각(통합)                    2,574
  감가상각비(개별)                         (없음)
  무형자산상각비(개별)                       (없음)
  사용권자산상각비                           576

📌 NWC
  매출채권                             3,863
  재고자산                             4,978
  매입채무                              (없음)

📌 CAPEX
  유형자산취득(CAPEX)                      810

📌 IBD 유동
  단기차입금                            3,062
  유동성장기차입금                          (없음)
  유동성사채(합산)                         (없음)
  유동리스부채                             613

📌 IBD 비유동
  장기차입금                             (없음)
  비유동사채(합산)                         (없음)
  비유동리스부채                            728

📌 NOA 후보
  현금및현금성자산                         4,515
  투자부동산                            5,947
  이연법인세자산                            509
  이연법인세부채   

In [33]:
# 셀 31: 커스텀 태그 포함 키워드 기반 실데이터 직접 탐색

print("=== '매입채무' 관련 tag 실데이터 전수 (키워드) ===\n")
ap_in_data = df[
    df["tag"].str.contains("Trade|Payable|payable|매입", na=False, case=False) &
    df["contextRef"].str.contains("CFY", na=False) &
    df["contextRef"].str.contains("Consolidated", na=False) &
    ~df["contextRef"].str.contains("Separate", na=False)
]
print(ap_in_data[["tag", "value", "contextRef"]].assign(
    value_억=lambda x: x["value"].astype(float)/1e8,
    ctx_끝=lambda x: x["contextRef"].str[-60:]
)[["tag","value_억","ctx_끝"]].to_string(index=False))

print("\n\n=== '확정급여/퇴직' 관련 tag 실데이터 전수 (키워드) ===\n")
db_in_data = df[
    df["tag"].str.contains("Benefit|Pension|Retire|Defined|retire", na=False, case=False) &
    df["contextRef"].str.contains("CFY", na=False) &
    df["contextRef"].str.contains("Consolidated", na=False) &
    ~df["contextRef"].str.contains("Separate", na=False)
]
print(db_in_data[["tag", "value", "contextRef"]].assign(
    value_억=lambda x: x["value"].astype(float)/1e8,
    ctx_끝=lambda x: x["contextRef"].str[-60:]
)[["tag","value_억","ctx_끝"]].to_string(index=False))

=== '매입채무' 관련 tag 실데이터 전수 (키워드) ===

Empty DataFrame
Columns: [tag, value_억, ctx_끝]
Index: []


=== '확정급여/퇴직' 관련 tag 실데이터 전수 (키워드) ===

Empty DataFrame
Columns: [tag, value_억, ctx_끝]
Index: []


In [34]:
# 셀 32: context 조건 완전 제거 — tag 키워드만으로 실데이터 전수 탐색

print("=== 매입채무 관련 tag 전수 (context 무관) ===\n")
ap_raw = df[df["tag"].str.contains("Pay|Trade|Payable", na=False, case=True)]
print(ap_raw["tag"].unique())

print("\n\n=== 확정급여 관련 tag 전수 (context 무관) ===\n")
db_raw = df[df["tag"].str.contains("Benefit|Pension|Defined|Retire", na=False, case=True)]
print(db_raw["tag"].unique())

print("\n\n=== 전체 tag 중 CFY+연결 context 고유값 확인 ===\n")
# 혹시 매입채무가 다른 이름으로 잡히는지 BS eFY context 전체 tag 목록
bs_all = df[
    df["contextRef"] == "CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"
]
print(f"단순 연결 BS context 항목 수: {len(bs_all)}")
print(bs_all[["tag","value"]].assign(
    value_억=lambda x: x["value"].astype(float)/1e8
).sort_values("value_억", ascending=False).to_string(index=False))

=== 매입채무 관련 tag 전수 (context 무관) ===

['IncreaseDecreaseThroughSharebasedPaymentTransactions'
 'OtherPayablesForRemainingSharesOfDisclosureOfAgreementBetweenShareholdersLineItemsOfDisclosureOfAgreementBetweenShareholdersTableOfItems'
 'PaymentsToAcquireSharesAndStockOptionOfDisclosureOfAgreementBetweenShareholdersLineItemsOfDisclosureOfAgreementBetweenShareholdersTableOfItems'
 'PaymentsToAcquireSharesAndStockOptionOfDisclosureOfAgreementBetweenShareholdersLineitemsOfDisclosureOfAgreementBetweenShareholdersTableOfItems'
 'AmountsPayableRelatedPartyTransactions'
 'ExpenseRelatingToVariableLeasePaymentsNotIncludedInMeasurementOfLeaseLiabilities'
 'UndiscountedFinanceLeasePaymentsToBeReceived'
 'NumberOfShareOptionsExercisableInCashsettledSharebasedPaymentArrangementOfDisclosureOfTermsAndConditionsOfShareBasedPaymentArrangementLineItemsOfDisclosureOfTermsAndConditionsOfSharebasedPaymentArrangementTableOfItems'
 'DescriptionOfMethodOfSettlementForSharebasedPaymentArrangement'
 'DateOfGrantO

In [35]:
# 셀 33: EXTRACTION_RULES_FINAL 최종 패치 — 실데이터 확인된 태그로 교체

EXTRACTION_RULES_FINAL.update({

    # 매입채무 — 실데이터 확인된 태그로 교체
    "매입채무": {
        "ctx": "BS",
        "tags": [
            "ShortTermTradePayables",                       # 세분화 기업
            "TradeAndOtherCurrentPayablesToTradeSuppliers", # 아모레 등
            "TradeAndOtherCurrentPayables",                 # 묶음
        ],
        "mode": "first"
    },

    # 확정급여 — 실데이터 확인된 태그로 교체
    "확정급여채무": {
        "ctx": "BS",
        "tags": [
            "DefinedBenefitObligationAtPresentValue",       # 총채무 (아모레)
            "NoncurrentRecognisedLiabilitiesDefinedBenefitPlan",  # 순부채
            "LiabilityAssetOfDefinedBenefitPlans",          # 순자산(-)
        ],
        "mode": "first"
    },
    "사외적립자산": {
        "ctx": "BS",
        "tags": [
            "NoncurrentRecognisedAssetsDefinedBenefitPlan", # 아모레
            "PlanAssetsAtFairValue",                        # 표준태그
            "FairValueOfPlanAssets",
        ],
        "mode": "first"
    },

    # 관계기업투자 (NOA 조건부)
    "관계기업투자": {
        "ctx": "BS",
        "tags": [
            "InvestmentAccountedForUsingEquityMethod",
            "InvestmentsInAssociates",
            "InvestmentsInSubsidiariesJointVenturesAndAssociates",
        ],
        "mode": "first"
    },
})

# 셀 34: 최종 재실행

raw = extract_final(df, EXTRACTION_RULES_FINAL)

def fmt(val):
    return f"{val/1e8:>12,.0f}" if val is not None else f"{'(없음)':>12}"

sections = {
    "📌 손익계산서": [
        "매출액","영업이익","당기순이익","지배주주순이익",
        "감가상각및무형상각(통합)","감가상각비(개별)",
        "무형자산상각비(개별)","사용권자산상각비"
    ],
    "📌 NWC":        ["매출채권","재고자산","매입채무"],
    "📌 CAPEX":      ["유형자산취득(CAPEX)"],
    "📌 IBD 유동":   ["단기차입금","유동성장기차입금","유동성사채(합산)","유동리스부채"],
    "📌 IBD 비유동": ["장기차입금","비유동사채(합산)","비유동리스부채"],
    "📌 NOA 후보":   [
        "현금및현금성자산","투자부동산",
        "이연법인세자산","이연법인세부채",
        "확정급여채무","사외적립자산",
        "관계기업투자"
    ],
}

print("=" * 58)
print("  아모레퍼시픽 2024 연결재무제표 (단위: 억원)")
print("=" * 58)
for section, items in sections.items():
    print(f"\n{section}")
    for item in items:
        print(f"  {item:28s} {fmt(raw.get(item))}")

# ── 계산 요약 ─────────────────────────────────────────────
print("\n" + "=" * 58)
print("  계산 요약")
print("=" * 58)

DA_integrated = raw.get("감가상각및무형상각(통합)")
DA_individual = sum(raw.get(k) or 0 for k in
                    ["감가상각비(개별)","무형자산상각비(개별)","사용권자산상각비"])
DA = DA_integrated if DA_integrated else DA_individual
DA_source = "통합태그" if DA_integrated else "개별합산"

EBIT  = raw.get("영업이익") or 0
NWC   = (raw.get("매출채권") or 0) + (raw.get("재고자산") or 0) - (raw.get("매입채무") or 0)
IBD   = sum(raw.get(k) or 0 for k in [
            "단기차입금","유동성장기차입금","유동성사채(합산)","유동리스부채",
            "장기차입금","비유동사채(합산)","비유동리스부채"])
CAPEX = abs(raw.get("유형자산취득(CAPEX)") or 0)

print(f"  EBIT      {EBIT/1e8:>10,.0f} 억원")
print(f"  D&A       {DA/1e8:>10,.0f} 억원  ({DA_source})")
print(f"  EBITDA    {(EBIT+DA)/1e8:>10,.0f} 억원")
print(f"  NWC       {NWC/1e8:>10,.0f} 억원  (매출채권+재고-매입채무)")
print(f"  IBD       {IBD/1e8:>10,.0f} 억원")
print(f"  CAPEX     {CAPEX/1e8:>10,.0f} 억원")

# DART 공시 대조
print("\n" + "=" * 58)
print("  DART 공시 대조")
print("=" * 58)
checks = {
    "매출액":     (raw.get("매출액"), 388514),
    "영업이익":   (raw.get("영업이익"), 22053),
    "D&A":       (DA, 257414),
    "매출채권":   (raw.get("매출채권"), 386340),
    "재고자산":   (raw.get("재고자산"), 497801),
    "매입채무":   (raw.get("매입채무"), 95965),
    "현금":       (raw.get("현금및현금성자산"), 451543),
}
for item, (extracted, dart_val) in checks.items():
    if extracted is None:
        print(f"  {item:12s}  추출: (없음)     DART: {dart_val/1e4:>8,.0f}억  ❌")
        continue
    ext_만 = extracted / 1e4  # 원 → 억 (DART는 백만원 단위)
    diff = abs(ext_만 - dart_val/1e4) / (dart_val/1e4) * 100
    flag = "✅" if diff < 1 else "⚠️"
    print(f"  {item:12s}  추출: {ext_만:>8,.0f}억  DART: {dart_val/1e4:>8,.0f}억  {flag} ({diff:.1f}%)")

  아모레퍼시픽 2024 연결재무제표 (단위: 억원)

📌 손익계산서
  매출액                                38,851
  영업이익                                2,205
  당기순이익                               6,016
  지배주주순이익                             5,932
  감가상각및무형상각(통합)                       2,574
  감가상각비(개별)                            (없음)
  무형자산상각비(개별)                          (없음)
  사용권자산상각비                              576

📌 NWC
  매출채권                                3,863
  재고자산                                4,978
  매입채무                                  960

📌 CAPEX
  유형자산취득(CAPEX)                         810

📌 IBD 유동
  단기차입금                               3,062
  유동성장기차입금                             (없음)
  유동성사채(합산)                            (없음)
  유동리스부채                                613

📌 IBD 비유동
  장기차입금                                (없음)
  비유동사채(합산)                            (없음)
  비유동리스부채                               728

📌 NOA 후보
  현금및현금성자산                            4,515
  투자부동산                           

In [36]:
# 셀 35: DART 대조 단위 수정

print("=" * 58)
print("  DART 공시 대조 (단위: 억원)")
print("=" * 58)

# 추출값: 원(KRW) → 억원: /1e8
# DART값: 백만원   → 억원: /100

checks = {
    "매출액":   (raw.get("매출액"),                  388514),   # 백만원
    "영업이익": (raw.get("영업이익"),                  22053),
    "D&A":     (DA,                                 257414),
    "매출채권": (raw.get("매출채권"),                  386340),
    "재고자산": (raw.get("재고자산"),                  497801),
    "매입채무": (raw.get("매입채무"),                   95965),
    "현금":     (raw.get("현금및현금성자산"),            451543),
}

for item, (extracted, dart_백만) in checks.items():
    dart_억 = dart_백만 / 100          # 백만원 → 억원
    if extracted is None:
        print(f"  {item:12s}  추출:     (없음)  DART: {dart_억:>8,.0f}억  ❌")
        continue
    ext_억 = extracted / 1e8           # 원 → 억원
    diff = abs(ext_억 - dart_억) / dart_억 * 100
    flag = "✅" if diff < 1 else "⚠️"
    print(f"  {item:12s}  추출: {ext_억:>8,.0f}억  DART: {dart_억:>8,.0f}억  {flag} ({diff:.1f}%)")

  DART 공시 대조 (단위: 억원)
  매출액           추출:   38,851억  DART:    3,885억  ⚠️ (900.0%)
  영업이익          추출:    2,205억  DART:      221억  ⚠️ (899.8%)
  D&A           추출:    2,574억  DART:    2,574억  ✅ (0.0%)
  매출채권          추출:    3,863억  DART:    3,863억  ✅ (0.0%)
  재고자산          추출:    4,978억  DART:    4,978억  ✅ (0.0%)
  매입채무          추출:      960억  DART:      960억  ✅ (0.0%)
  현금            추출:    4,515억  DART:    4,515억  ✅ (0.0%)


In [37]:
# 셀 36: DART 기준값 수정 — 백만원 단위 올바르게 입력

print("=" * 58)
print("  DART 공시 대조 (단위: 억원)")
print("=" * 58)

checks = {
    "매출액":   (raw.get("매출액"),               3885140),  # 백만원: 3,885,140
    "영업이익": (raw.get("영업이익"),               220528),  # 백만원: 220,528
    "D&A":     (DA,                              257414),
    "매출채권": (raw.get("매출채권"),               386340),
    "재고자산": (raw.get("재고자산"),               497801),
    "매입채무": (raw.get("매입채무"),                95965),
    "현금":     (raw.get("현금및현금성자산"),         451543),
}

for item, (extracted, dart_백만) in checks.items():
    dart_억 = dart_백만 / 100
    if extracted is None:
        print(f"  {item:12s}  추출:     (없음)  DART: {dart_억:>8,.0f}억  ❌")
        continue
    ext_억 = extracted / 1e8
    diff = abs(ext_억 - dart_억) / dart_억 * 100
    flag = "✅" if diff < 1 else "⚠️"
    print(f"  {item:12s}  추출: {ext_억:>8,.0f}억  DART: {dart_억:>8,.0f}억  {flag} ({diff:.1f}%)")

  DART 공시 대조 (단위: 억원)
  매출액           추출:   38,851억  DART:   38,851억  ✅ (0.0%)
  영업이익          추출:    2,205억  DART:    2,205억  ✅ (0.0%)
  D&A           추출:    2,574억  DART:    2,574억  ✅ (0.0%)
  매출채권          추출:    3,863억  DART:    3,863억  ✅ (0.0%)
  재고자산          추출:    4,978억  DART:    4,978억  ✅ (0.0%)
  매입채무          추출:      960억  DART:      960억  ✅ (0.0%)
  현금            추출:    4,515억  DART:    4,515억  ✅ (0.0%)


In [38]:
# 셀 37: NWC / CAPEX / IBD 타당성 검증

print("=" * 65)
print("  아모레퍼시픽 2024 타당성 검증 (단위: 억원)")
print("=" * 65)

# ── 단위 변환 헬퍼 ───────────────────────────────────────
def 억(val):
    return val / 1e8 if val is not None else 0

# ══════════════════════════════════════════════════════════
# 1. NWC
# ══════════════════════════════════════════════════════════
매출채권 = 억(raw.get("매출채권"))
재고자산  = 억(raw.get("재고자산"))
매입채무  = 억(raw.get("매입채무"))
NWC      = 매출채권 + 재고자산 - 매입채무

print("\n【 NWC (Operating Net Working Capital) 】")
print(f"  매출채권              {매출채권:>10,.0f}  ← CurrentTradeReceivables")
print(f"  재고자산              {재고자산:>10,.0f}  ← Inventories")
print(f"  매입채무             -{매입채무:>10,.0f}  ← TradeAndOtherCurrentPayablesToTradeSuppliers")
print(f"  {'─'*45}")
print(f"  NWC                  {NWC:>10,.0f}")
print(f"\n  [검증] 매출액 대비 NWC 비율: {NWC/억(raw.get('매출액'))*100:.1f}%")
print(f"         (화장품 업종 통상 10~25% → ", end="")
print("적정 ✅" if 10 <= NWC/억(raw.get('매출액'))*100 <= 25 else "범위 외 ⚠️", end=")\n")

# ══════════════════════════════════════════════════════════
# 2. CAPEX
# ══════════════════════════════════════════════════════════
capex = abs(억(raw.get("유형자산취득(CAPEX)")))
매출액 = 억(raw.get("매출액"))
ebitda = 억(raw.get("영업이익")) + 억(DA_integrated or DA_individual)

print("\n【 CAPEX 】")
print(f"  유형자산취득          {capex:>10,.0f}  ← CF1 PurchaseOfPPE")
print(f"\n  [검증] 매출액 대비 CAPEX: {capex/매출액*100:.1f}%")
print(f"         (화장품 업종 통상 1~5% → ", end="")
print("적정 ✅" if 1 <= capex/매출액*100 <= 5 else "범위 외 ⚠️", end=")\n")
print(f"  [검증] EBITDA 대비 CAPEX: {capex/ebitda*100:.1f}%")
print(f"         (통상 20% 이하 → ", end="")
print("적정 ✅" if capex/ebitda*100 <= 20 else "범위 외 ⚠️", end=")\n")

# ══════════════════════════════════════════════════════════
# 3. IBD
# ══════════════════════════════════════════════════════════
단기차입금     = 억(raw.get("단기차입금"))
유동성장기차입금 = 억(raw.get("유동성장기차입금"))
유동성사채     = 억(raw.get("유동성사채(합산)"))
유동리스부채   = 억(raw.get("유동리스부채"))
장기차입금     = 억(raw.get("장기차입금"))
비유동사채     = 억(raw.get("비유동사채(합산)"))
비유동리스부채  = 억(raw.get("비유동리스부채"))

IBD_유동  = 단기차입금 + 유동성장기차입금 + 유동성사채 + 유동리스부채
IBD_비유동 = 장기차입금 + 비유동사채 + 비유동리스부채
IBD_total = IBD_유동 + IBD_비유동

print("\n【 IBD (Interest-Bearing Debt) 】")
print(f"  [유동]")
print(f"    단기차입금          {단기차입금:>10,.0f}  ← ShorttermBorrowings")
print(f"    유동성장기차입금     {유동성장기차입금:>10,.0f}  ← CurrentPortionOfLongtermBorrowings")
print(f"    유동성사채          {유동성사채:>10,.0f}  ← 전환/신주/교환사채 합산")
print(f"    유동리스부채        {유동리스부채:>10,.0f}  ← CurrentLeaseLiabilities")
print(f"    소계               {IBD_유동:>10,.0f}")
print(f"\n  [비유동]")
print(f"    장기차입금          {장기차입금:>10,.0f}  ← LongTermBorrowingsGross")
print(f"    비유동사채          {비유동사채:>10,.0f}  ← BondsIssued 등 합산")
print(f"    비유동리스부채       {비유동리스부채:>10,.0f}  ← NoncurrentLeaseLiabilities")
print(f"    소계               {IBD_비유동:>10,.0f}")
print(f"  {'─'*45}")
print(f"  IBD 합계             {IBD_total:>10,.0f}")

# IBD 검증
부채총계 = 억(raw.get("Liabilities")) if raw.get("Liabilities") else None
자본총계  = 억(raw.get("Equity"))      if raw.get("Equity")      else None

# 부채총계/자본총계는 EXTRACTION_RULES에 있으므로 직접 추출
liabilities = get_value(df, "Liabilities", "BS") 
equity      = get_value(df, "Equity",      "BS")

if liabilities and equity:
    부채총계_억 = liabilities / 1e8
    자본총계_억 = equity / 1e8
    부채비율    = IBD_total / 자본총계_억 * 100
    이자커버리지 = ebitda / (IBD_total * 0.04)  # 가정 금리 4%
    print(f"\n  [검증] IBD/자본총계(부채비율): {부채비율:.1f}%")
    print(f"         (화장품 우량기업 통상 50% 이하 → ", end="")
    print("적정 ✅" if 부채비율 <= 50 else "범위 외 ⚠️", end=")\n")
    print(f"  [검증] EBITDA/이자추정(이자커버리지): {이자커버리지:.1f}x")
    print(f"         (통상 3x 이상 → ", end="")
    print("적정 ✅" if 이자커버리지 >= 3 else "위험 ⚠️", end=")\n")

# DART 재무상태표 교차검증
print(f"\n  [교차검증] IBD vs DART 재무상태표 차입금")
print(f"  LiabilitiesArisingFromFinancingActivities 확인:")
laf = df[
    (df["tag"] == "LiabilitiesArisingFromFinancingActivities") &
    (df["contextRef"] == "CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember")
]
if not laf.empty:
    laf_억 = int(laf.iloc[0]["value"]) / 1e8
    print(f"    재무활동부채 합계: {laf_억:,.0f}억  vs  IBD: {IBD_total:,.0f}억")
    diff = abs(laf_억 - IBD_total) / laf_억 * 100
    print(f"    차이: {diff:.1f}% → ", end="")
    print("✅ 일치" if diff < 5 else "⚠️ 확인 필요")

  아모레퍼시픽 2024 타당성 검증 (단위: 억원)

【 NWC (Operating Net Working Capital) 】
  매출채권                   3,863  ← CurrentTradeReceivables
  재고자산                   4,978  ← Inventories
  매입채무             -       960  ← TradeAndOtherCurrentPayablesToTradeSuppliers
  ─────────────────────────────────────────────
  NWC                       7,882

  [검증] 매출액 대비 NWC 비율: 20.3%
         (화장품 업종 통상 10~25% → 적정 ✅)

【 CAPEX 】
  유형자산취득                 810  ← CF1 PurchaseOfPPE

  [검증] 매출액 대비 CAPEX: 2.1%
         (화장품 업종 통상 1~5% → 적정 ✅)
  [검증] EBITDA 대비 CAPEX: 16.9%
         (통상 20% 이하 → 적정 ✅)

【 IBD (Interest-Bearing Debt) 】
  [유동]
    단기차입금               3,062  ← ShorttermBorrowings
    유동성장기차입금              0  ← CurrentPortionOfLongtermBorrowings
    유동성사채                   0  ← 전환/신주/교환사채 합산
    유동리스부채               613  ← CurrentLeaseLiabilities
    소계                    3,675

  [비유동]
    장기차입금                   0  ← LongTermBorrowingsGross
    비유동사채                   0  ← BondsIssued 등 합산
    비유동리스부채

In [39]:
# 셀 38: 단순 연결 BS context 전체 항목 NOA 분류

# 설계서 v4 기준 분류 규칙
NOA_RULES = {

    # ── 확실한 NOA (업종 무관) ──────────────────────────────
    "DEFINITE_NOA": {
        "tags": [
            "InvestmentProperty",                          # 투자부동산
        ],
        "label": "확실한 NOA"
    },

    # ── 기본 NOA (업종 예외 적용) ────────────────────────────
    "DEFAULT_NOA": {
        "tags": [
            "CashAndCashEquivalents",                      # 현금및현금성자산
            "ShorttermDepositsNotClassifiedAsCashEquivalents",  # 단기금융상품
            "LongtermDeposits",                            # 장기금융상품
            "FinancialAssetsAtFairValueThroughProfitOrLoss",    # FVTPL금융자산
            "CurrentFinancialAssetsAtFairValueThroughProfitOrLoss",
            "NoncurrentFinancialAssetsAtFairValueThroughProfitOrLoss",
            "FinancialAssetsAtFairValueThroughOtherComprehensiveIncome",  # FVOCI
            "FinancialAssetsAtAmortisedCost",              # 상각후원가금융자산
            "InvestmentAccountedForUsingEquityMethod",     # 관계기업투자
        ],
        "label": "기본 NOA"
    },

    # ── 조건부 NOA ───────────────────────────────────────────
    "COND_NOA": {
        "tags": [
            "DeferredTaxAssets",                           # 이연법인세자산 (Damodaran=NOA)
            "DeferredTaxLiabilities",                      # 이연법인세부채 (차감)
            "NoncurrentRecognisedAssetsDefinedBenefitPlan",# 사외적립자산
            "DefinedBenefitObligationAtPresentValue",      # 확정급여채무 (차감)
            "NoncurrentRecognisedLiabilitiesDefinedBenefitPlan",
        ],
        "label": "조건부 NOA"
    },

    # ── OA로 처리 (NOA 아님) ─────────────────────────────────
    "OA": {
        "tags": [
            "CurrentTradeReceivables",                     # 매출채권
            "Inventories",                                 # 재고자산
            "OtherCurrentReceivables",                     # 기타유동채권
            "OtherCurrentAssets",                          # 기타유동자산
            "CurrentTaxAssets",                            # 당기법인세자산
            "TradeAndOtherCurrentPayablesToTradeSuppliers",# 매입채무
            "OtherCurrentPayables",                        # 기타유동채무
            "CurrentTaxLiabilities",                       # 당기법인세부채
            "CurrentContractLiabilities",                  # 계약부채
            "CurrentProvisions",                           # 유동충당부채
            "OtherCurrentLiabilities",                     # 기타유동부채
            "OtherCurrentFinancialLiabilities",            # 기타유동금융부채
        ],
        "label": "영업자산(OA)"
    },

    # ── IBD (이미 별도 추출) ──────────────────────────────────
    "IBD": {
        "tags": [
            "ShorttermBorrowings",
            "CurrentLeaseLiabilities",
            "NoncurrentLeaseLiabilities",
            "GrossLeaseLiabilities",
            "LeaseLiabilities",
            "LiabilitiesArisingFromFinancingActivities",
        ],
        "label": "IBD(별도처리)"
    },

    # ── 별도 처리 (NOA/OA 아님) ──────────────────────────────
    "SEPARATE": {
        "tags": [
            "Goodwill",                                    # 영업권 (EV에 반영)
            "RightofuseAssets",                            # 사용권자산 (IBD 대응)
            "GovernmentGrants",                            # 정부보조금
        ],
        "label": "별도처리"
    },
}

# 역방향 매핑: tag → 분류
tag_to_class = {}
for cls, info in NOA_RULES.items():
    for tag in info["tags"]:
        tag_to_class[tag] = (cls, info["label"])

# 단순 연결 BS context 전체 항목에 분류 적용
CTX_PLAIN_BS = "CFY2024eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"

bs_all = df[df["contextRef"] == CTX_PLAIN_BS].copy()
bs_all["value_억"] = bs_all["value"].astype(float) / 1e8
bs_all["분류"] = bs_all["tag"].map(lambda t: tag_to_class.get(t, ("UNCLASSIFIED", "미분류"))[1])
bs_all["분류코드"] = bs_all["tag"].map(lambda t: tag_to_class.get(t, ("UNCLASSIFIED", "?"))[0])

# 분류별 출력
order = ["확실한 NOA", "기본 NOA", "조건부 NOA", "영업자산(OA)", "IBD(별도처리)", "별도처리", "미분류"]
print("=" * 70)
print("  아모레퍼시픽 2024 BS 항목 NOA 분류 결과")
print("=" * 70)
for label in order:
    subset = bs_all[bs_all["분류"] == label].sort_values("value_억", ascending=False)
    if subset.empty:
        continue
    total = subset["value_억"].sum()
    print(f"\n【 {label} 】  소계: {total:,.0f}억")
    for _, r in subset.iterrows():
        kor = tag_to_kor.get(r["tag"], r["tag"][:40])
        print(f"  {kor:35s} {r['value_억']:>10,.0f}  ({r['tag'][:45]})")

  아모레퍼시픽 2024 BS 항목 NOA 분류 결과

【 확실한 NOA 】  소계: 5,947억
  투자부동산                                    5,947  (InvestmentProperty)

【 기본 NOA 】  소계: 10,600억
  현금및현금성자산                                 4,515  (CashAndCashEquivalents)
  FinancialAssetsAtFairValueThroughProfitO      2,575  (FinancialAssetsAtFairValueThroughProfitOrLoss)
  CurrentFinancialAssetsAtFairValueThrough      2,265  (CurrentFinancialAssetsAtFairValueThroughProfi)
  ShorttermDepositsNotClassifiedAsCashEqui        467  (ShorttermDepositsNotClassifiedAsCashEquivalen)
  FinancialAssetsAtAmortisedCost             310  (FinancialAssetsAtAmortisedCost)
  LongtermDeposits                           265  (LongtermDeposits)
  NoncurrentFinancialAssetsAtFairValueThro         95  (NoncurrentFinancialAssetsAtFairValueThroughPr)
  지분법적용 투자지분                                  58  (InvestmentAccountedForUsingEquityMethod)
  FinancialAssetsAtFairValueThroughOtherCo         51  (FinancialAssetsAtFairValueThroughOtherCompreh)

【 조건부 NOA 】  소

In [41]:
# 셀 39 수정: 상위집계 태그 제거로 중복 해소

EXCLUDE_TAGS_UPDATED = EXCLUDE_TAGS | {
    # 하위항목 합계 → 상위태그 제거, 세부태그만 사용
    "FinancialAssets",
    "FinancialAssetsAtFairValueThroughProfitOrLoss",   # Current/Noncurrent 합계
    "FinancialAssetsAtAmortisedCost",                  # Current/Noncurrent 합계
    "FinancialAssetsAtFairValueThroughOtherComprehensiveIncome",  # 하위 있으면 제거
}

NOA_RULES_FINAL_V2 = {
    "DEFINITE_NOA": [
        "InvestmentProperty",
    ],
    "DEFAULT_NOA": [
        "CashAndCashEquivalents",
        "ShorttermDepositsNotClassifiedAsCashEquivalents",
        "LongtermDeposits",
        # FVTPL: Current/Noncurrent 세부만
        "CurrentFinancialAssetsAtFairValueThroughProfitOrLoss",
        "NoncurrentFinancialAssetsAtFairValueThroughProfitOrLoss",
        # FVOCI: 세부만
        "NoncurrentInvestmentsInEquityInstrumentsDesignatedAtFairValueThroughOtherComprehensiveIncome",
        # AC: Current/Noncurrent 세부만
        "CurrentFinancialAssetsAtAmortisedCost",
        "NoncurrentFinancialAssetsAtAmortisedCost",
        "InvestmentAccountedForUsingEquityMethod",
        "OtherNoncurrentReceivables",
        "OtherNoncurrentAssets",
    ],
    "COND_NOA": [
        "DeferredTaxAssets",
        ("DeferredTaxLiabilities",                         -1),
        "NoncurrentRecognisedAssetsDefinedBenefitPlan",
        ("NoncurrentRecognisedLiabilitiesDefinedBenefitPlan", -1),
        ("DefinedBenefitObligationAtPresentValue",         -1),
        ("OtherNoncurrentLiabilities",                     -1),
        ("OtherNoncurrentFinancialLiabilities",            -1),
        ("NoncurrentProvisions",                           -1),
    ],
}

noa_items_v2, total_noa_v2 = calc_noa(
    df, NOA_RULES_FINAL_V2, CTX_BS, EXCLUDE_TAGS_UPDATED
)

# 출력
print("=" * 65)
print("  아모레퍼시픽 2024 NOA 분류 결과 v2 (단위: 억원)")
print("=" * 65)

cls_label = {
    "DEFINITE_NOA": "확실한 NOA",
    "DEFAULT_NOA":  "기본 NOA",
    "COND_NOA":     "조건부 NOA",
}
for cls_key, cls_name in cls_label.items():
    items = [(t, k, v) for c, t, k, v in noa_items_v2 if c == cls_key]
    if not items:
        continue
    subtotal = sum(v for _, _, v in items)
    print(f"\n【 {cls_name} 】  소계: {subtotal:,.0f}억")
    for tag, kor, val in items:
        sign_str = "(-)" if val < 0 else "   "
        print(f"  {sign_str} {kor:32s} {val:>10,.0f}")

print(f"\n{'='*65}")
print(f"  NOA 합계                                {total_noa_v2:>10,.0f} 억원")

# 타당성 검증
자산총계_억 = get_value(df, "Assets", "BS") / 1e8
print(f"\n  [검증] NOA / 자산총계: {total_noa_v2/자산총계_억*100:.1f}%")
print(f"         (통상 10~30% → ", end="")
print("적정 ✅" if 10 <= total_noa_v2/자산총계_억*100 <= 30 else "범위 외 ⚠️", end=")\n")

  아모레퍼시픽 2024 NOA 분류 결과 v2 (단위: 억원)

【 확실한 NOA 】  소계: 5,947억
      투자부동산                                 5,947

【 기본 NOA 】  소계: 8,697억
      현금및현금성자산                              4,515
      ShorttermDepositsNotClassifiedAsCas        467
      LongtermDeposits                        265
      CurrentFinancialAssetsAtFairValueTh      2,265
      NoncurrentFinancialAssetsAtFairValu         95
      유동 상각후원가 측정 금융자산                        300
      비유동 상각후원가 측정 금융자산                        10
      지분법적용 투자지분                               58
      OtherNoncurrentReceivables              638
      OtherNoncurrentAssets                    83

【 조건부 NOA 】  소계: -6,814억
      이연법인세자산                                 509
  (-) 이연법인세부채                              -2,358
      NoncurrentRecognisedAssetsDefinedBe        754
  (-) NoncurrentRecognisedLiabilitiesDefi        -63
  (-) DefinedBenefitObligationAtPresentVa     -5,007
  (-) OtherNoncurrentLiabilities             -334
  (-) 기타비유동금융부채      

In [43]:
# 셀 40 수정: def억 제거

def get_nwc(df, year):
    매출채권 = get_value(df, "CurrentTradeReceivables",                     "BS", year) or \
               get_value(df, "ShortTermTradeReceivable",                    "BS", year) or \
               get_value(df, "TradeAndOtherCurrentReceivables",             "BS", year)

    재고자산  = get_value(df, "Inventories", "BS", year)

    매입채무  = get_value(df, "TradeAndOtherCurrentPayablesToTradeSuppliers", "BS", year) or \
               get_value(df, "ShortTermTradePayables",                      "BS", year) or \
               get_value(df, "TradeAndOtherCurrentPayables",                "BS", year)

    items = {"매출채권": 매출채권, "재고자산": 재고자산, "매입채무": 매입채무}
    nwc = (매출채권 or 0) + (재고자산 or 0) - (매입채무 or 0)
    return items, nwc

# 당기/전기 NWC 동시 추출
items_cur, nwc_cur = get_nwc(df, "CFY2024")
items_pfy, nwc_pfy = get_nwc(df, "PFY2023")
delta_nwc = nwc_cur - nwc_pfy

print("=" * 60)
print("  NWC 변동 분석 (단위: 억원)")
print("=" * 60)
print(f"\n{'항목':15s} {'전기(2023)':>12} {'당기(2024)':>12} {'변동':>10}")
print(f"{'─'*52}")
for key in ["매출채권", "재고자산", "매입채무"]:
    pfy_v = 억(items_pfy.get(key))
    cur_v = 억(items_cur.get(key))
    print(f"  {key:13s} {pfy_v:>12,.0f} {cur_v:>12,.0f} {cur_v-pfy_v:>+10,.0f}")

print(f"{'─'*52}")
print(f"  {'NWC 합계':13s} {억(nwc_pfy):>12,.0f} {억(nwc_cur):>12,.0f} {억(delta_nwc):>+10,.0f}")
print(f"\n  ΔNWC = {억(delta_nwc):+,.0f}억원")
print(f"  ({'NWC 증가 → 현금유출 ↑' if delta_nwc > 0 else 'NWC 감소 → 현금유입 ↑'})")

매출액_pfy = get_value(df, "Revenue", "IS", "PFY2023")
매출액_cur = get_value(df, "Revenue", "IS", "CFY2024")
print(f"\n  [검증] 전기 NWC/매출액: {억(nwc_pfy)/억(매출액_pfy)*100:.1f}%")
print(f"  [검증] 당기 NWC/매출액: {억(nwc_cur)/억(매출액_cur)*100:.1f}%")

  NWC 변동 분석 (단위: 억원)

항목                  전기(2023)     당기(2024)         변동
────────────────────────────────────────────────────
  매출채권                 2,929        3,863       +934
  재고자산                 3,943        4,978     +1,035
  매입채무                   864          960        +95
────────────────────────────────────────────────────
  NWC 합계               6,008        7,882     +1,874

  ΔNWC = +1,874억원
  (NWC 증가 → 현금유출 ↑)

  [검증] 전기 NWC/매출액: 16.4%
  [검증] 당기 NWC/매출액: 20.3%


In [44]:
# 셀 41: 순확정급여자산/부채 실데이터 태그 확인

target = [
    "NetDefinedBenefitAsset",
    "NetDefinedBenefitLiability",
    "NoncurrentRecognisedAssetsDefinedBenefitPlan",    # 기존 사용
    "NoncurrentRecognisedLiabilitiesDefinedBenefitPlan",
    "RecognisedLiabilitiesDefinedBenefitPlan",
    "RecognisedAssetsDefinedBenefitPlan",
]

print("=== 순확정급여 관련 태그 실데이터 확인 ===\n")
for tag in target:
    matched = df[df["tag"] == tag]
    if matched.empty:
        print(f"❌ {tag}")
        continue
    cfy = matched[
        matched["contextRef"].str.contains("CFY", na=False) &
        matched["contextRef"].str.contains("Consolidated", na=False) &
        ~matched["contextRef"].str.contains("Separate", na=False)
    ]
    print(f"✅ {tag}")
    for _, r in cfy.iterrows():
        val = int(r["value"]) / 1e8
        plain = "★단순" if r["contextRef"].endswith("ConsolidatedMember") else "  기타"
        print(f"   {plain}  {val:>8,.1f}억  |  {r['contextRef'][-50:]}")
    print()

=== 순확정급여 관련 태그 실데이터 확인 ===

❌ NetDefinedBenefitAsset
❌ NetDefinedBenefitLiability
✅ NoncurrentRecognisedAssetsDefinedBenefitPlan

✅ NoncurrentRecognisedLiabilitiesDefinedBenefitPlan

❌ RecognisedLiabilitiesDefinedBenefitPlan
❌ RecognisedAssetsDefinedBenefitPlan


In [ ]:
# 셀 42: NOA_RULES 최종 확정 (v3)

NOA_RULES_FINAL_V3 = {
    "DEFINITE_NOA": [
        "InvestmentProperty",                          # 투자부동산
    ],
    "DEFAULT_NOA": [
        "CashAndCashEquivalents",                      # 현금및현금성자산
        "ShorttermDepositsNotClassifiedAsCashEquivalents",  # 단기금융기관예치금
        "LongtermDeposits",                            # 장기금융기관예치금
        "CurrentFinancialAssetsAtFairValueThroughProfitOrLoss",   # 유동 FVTPL
        "NoncurrentFinancialAssetsAtFairValueThroughProfitOrLoss",# 비유동 FVTPL
        "CurrentFinancialAssetsAtAmortisedCost",       # 유동 AC금융자산
        "NoncurrentFinancialAssetsAtAmortisedCost",    # 비유동 AC금융자산
        "NoncurrentInvestmentsInEquityInstrumentsDesignatedAtFairValueThroughOtherComprehensiveIncome",  # FVOCI
        "InvestmentAccountedForUsingEquityMethod",     # 관계기업투자주식
    ],
    "COND_NOA": [
        # 순확정급여자산/부채 (재무상태표 표시 기준)
        "NoncurrentRecognisedAssetsDefinedBenefitPlan",          # 순확정급여자산 (+)
        ("NoncurrentRecognisedLiabilitiesDefinedBenefitPlan", -1),# 순확정급여부채 (-)
        # 기타비유동부채 계열 (비영업성)
        ("OtherNoncurrentLiabilities",          -1),   # 기타비유동부채
        ("OtherNoncurrentFinancialLiabilities", -1),   # 기타비유동금융부채
        ("NoncurrentProvisions",                -1),   # 비유동충당부채
    ],
}

def calc_noa_v3(df, rules, ctx_bs, year="CFY2024"):
    """NOA 계산 — 재무상태표 기준 순액"""
    noa_items = []
    total = 0

    label_map = {
        "DEFINITE_NOA": "확실한 NOA",
        "DEFAULT_NOA":  "기본 NOA",
        "COND_NOA":     "조건부 NOA",
    }
    kor_map = {
        "InvestmentProperty":               "투자부동산",
        "CashAndCashEquivalents":           "현금및현금성자산",
        "ShorttermDepositsNotClassifiedAsCashEquivalents": "단기금융기관예치금",
        "LongtermDeposits":                 "장기금융기관예치금",
        "CurrentFinancialAssetsAtFairValueThroughProfitOrLoss":    "유동 FVTPL금융자산",
        "NoncurrentFinancialAssetsAtFairValueThroughProfitOrLoss": "비유동 FVTPL금융자산",
        "CurrentFinancialAssetsAtAmortisedCost":   "유동 AC금융자산",
        "NoncurrentFinancialAssetsAtAmortisedCost":"비유동 AC금융자산",
        "NoncurrentInvestmentsInEquityInstrumentsDesignatedAtFairValueThroughOtherComprehensiveIncome": "FVOCI금융자산",
        "InvestmentAccountedForUsingEquityMethod": "관계기업투자주식",
        "NoncurrentRecognisedAssetsDefinedBenefitPlan":    "순확정급여자산",
        "NoncurrentRecognisedLiabilitiesDefinedBenefitPlan":"순확정급여부채",
        "OtherNoncurrentLiabilities":          "기타비유동부채",
        "OtherNoncurrentFinancialLiabilities":  "기타비유동금융부채",
        "NoncurrentProvisions":                "비유동충당부채",
    }

    for cls, tags in rules.items():
        for tag_item in tags:
            sign = 1
            tag = tag_item if isinstance(tag_item, str) else tag_item[0]
            if isinstance(tag_item, tuple):
                sign = tag_item[1]

            val = get_value(df, tag, "BS", year)
            if val is None:
                continue
            val_억 = val / 1e8 * sign
            total += val_억
            noa_items.append({
                "cls": label_map[cls],
                "tag": tag,
                "kor": kor_map.get(tag, tag[:35]),
                "val": val_억,
            })

    return noa_items, total

In [47]:
# 셀 43: 전체 파이프라인 함수화

def get_financials(df, year_cur="CFY2024", year_pfy="PFY2023"):
    """
    XBRL DataFrame → 재무지표 전체 추출
    Returns: dict
    """
    CTX_IS  = f"{year_cur}dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"
    CTX_BS  = f"{year_cur}eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"
    CTX_CF  = CTX_IS

    def g(tag, ctx="BS", year=year_cur):
        return get_value(df, tag, ctx, year)

    # ── 손익계산서 ───────────────────────────────────────
    매출액   = g("Revenue",           "IS")
    영업이익  = g("OperatingIncomeLoss","IS")
    당기순이익 = g("ProfitLoss",        "IS")
    지배순이익 = g("ProfitLossAttributableToOwnersOfParent","IS")

    DA_통합  = g("DepreciationAndAmortisationExpense","IS")
    DA_유형  = g("DepreciationExpense",               "IS")
    DA_무형  = g("AmortisationExpense",               "IS")
    DA_사용권 = g("DepreciationRightofuseAssets",      "IS")
    DA = DA_통합 if DA_통합 else sum(x or 0 for x in [DA_유형, DA_무형, DA_사용권])

    # ── NWC 당기/전기 ────────────────────────────────────
    def _nwc(year):
        ar = (g("CurrentTradeReceivables",                     "BS", year) or
              g("ShortTermTradeReceivable",                    "BS", year) or
              g("TradeAndOtherCurrentReceivables",             "BS", year))
        inv = g("Inventories", "BS", year)
        ap  = (g("TradeAndOtherCurrentPayablesToTradeSuppliers","BS", year) or
               g("ShortTermTradePayables",                     "BS", year) or
               g("TradeAndOtherCurrentPayables",               "BS", year))
        nwc = (ar or 0) + (inv or 0) - (ap or 0)
        return {"매출채권": ar, "재고자산": inv, "매입채무": ap, "NWC": nwc}

    nwc_cur = _nwc(year_cur)
    nwc_pfy = _nwc(year_pfy)
    delta_nwc = nwc_cur["NWC"] - nwc_pfy["NWC"]

    # ── CAPEX ────────────────────────────────────────────
    capex = g("PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities","CF")

    # ── IBD ─────────────────────────────────────────────
    ibd_tags = {
        "단기차입금":      ("ShorttermBorrowings",                  "BS"),
        "유동성장기차입금": ("CurrentPortionOfLongtermBorrowings",    "BS"),
        "유동리스부채":    ("CurrentLeaseLiabilities",               "BS"),
        "비유동리스부채":  ("NoncurrentLeaseLiabilities",            "BS"),
    }
    ibd_사채_유동 = sum(g(t,"BS") or 0 for t in [
        "CurrentPortionOfBonds","CurrentPortionOfConvertibleBonds",
        "CurrentPortionOfBondWithWarrant","CurrentPortionOfExchangeableBond"])
    ibd_사채_비유동 = sum(g(t,"BS") or 0 for t in [
        "BondsIssued","ConvertibleBonds","BondWithWarrant","ExchangeableBonds"])

    ibd_detail = {k: g(v[0], v[1]) for k, v in ibd_tags.items()}
    ibd_detail["유동성사채"] = ibd_사채_유동 or None
    ibd_detail["비유동사채"] = ibd_사채_비유동 or None
    IBD = sum(v or 0 for v in ibd_detail.values())

    # ── NOA ─────────────────────────────────────────────
    noa_items, NOA = calc_noa_v3(df, NOA_RULES_FINAL_V3, CTX_BS, year_cur)

    # ── 계산 지표 ────────────────────────────────────────
    EBITDA = (영업이익 or 0) + (DA or 0)
    FCFF   = (영업이익 or 0) * (1 - 0.22) + (DA or 0) - abs(capex or 0) - delta_nwc

    return {
        "year": year_cur,
        # 손익
        "매출액":    매출액,
        "영업이익":  영업이익,
        "당기순이익": 당기순이익,
        "지배순이익": 지배순이익,
        "DA":        DA,
        "EBITDA":    EBITDA,
        # NWC
        "nwc_cur":   nwc_cur,
        "nwc_pfy":   nwc_pfy,
        "delta_nwc": delta_nwc,
        # CAPEX
        "CAPEX":     capex,
        # IBD
        "IBD":       IBD,
        "ibd_detail":ibd_detail,
        # NOA
        "NOA":       NOA * 1e8,   # 억원 → 원으로 통일
        "noa_items": noa_items,
        # FCFF (참고용, 세율 22% 가정)
        "FCFF":      FCFF,
    }

In [50]:
# 셀 44 수정: billions 함수명 사용

result = get_financials(df)

def billions(val):
    return (val or 0) / 1e8

print("=" * 60)
print(f"  아모레퍼시픽 {result['year']} 연결 재무지표 (억원)")
print("=" * 60)

print("\n【 손익계산서 】")
print(f"  매출액           {billions(result['매출액']):>10,.0f}")
print(f"  영업이익(EBIT)   {billions(result['영업이익']):>10,.0f}")
print(f"  D&A              {billions(result['DA']):>10,.0f}")
print(f"  EBITDA           {billions(result['EBITDA']):>10,.0f}")
print(f"  당기순이익        {billions(result['당기순이익']):>10,.0f}")
print(f"  지배순이익        {billions(result['지배순이익']):>10,.0f}")

print("\n【 NWC 변동 】")
print(f"  {'항목':10s} {'전기':>10} {'당기':>10} {'변동':>10}")
print(f"  {'─'*43}")
for k in ["매출채권","재고자산","매입채무"]:
    pfy = billions(result['nwc_pfy'][k])
    cur = billions(result['nwc_cur'][k])
    print(f"  {k:10s} {pfy:>10,.0f} {cur:>10,.0f} {cur-pfy:>+10,.0f}")
print(f"  {'─'*43}")
nwc_pfy = billions(result['nwc_pfy']['NWC'])
nwc_cur = billions(result['nwc_cur']['NWC'])
delta   = billions(result['delta_nwc'])
print(f"  {'NWC':10s} {nwc_pfy:>10,.0f} {nwc_cur:>10,.0f} {delta:>+10,.0f}")
print(f"  → ΔNWC = {delta:+,.0f}억  ({'현금유출↑' if delta > 0 else '현금유입↑'})")

print("\n【 CAPEX 】")
print(f"  유형자산취득      {abs(billions(result['CAPEX'])):>10,.0f}")

print("\n【 IBD 】")
for k, v in result['ibd_detail'].items():
    if v:
        print(f"  {k:14s}  {billions(v):>10,.0f}")
print(f"  {'─'*27}")
print(f"  IBD 합계          {billions(result['IBD']):>10,.0f}")

print("\n【 NOA 】")
for cls in ["확실한 NOA","기본 NOA","조건부 NOA"]:
    items = [x for x in result['noa_items'] if x['cls'] == cls]
    if not items:
        continue
    sub = sum(x['val'] for x in items)
    print(f"  [{cls}]  소계: {sub:,.0f}억")
    for x in items:
        sign = "(-)" if x['val'] < 0 else "   "
        print(f"    {sign} {x['kor']:25s} {x['val']:>8,.0f}")
print(f"  {'─'*35}")
print(f"  NOA 합계          {billions(result['NOA']):>10,.0f}")

print("\n【 FCFF (참고, 세율 22% 가정) 】")
print(f"  FCFF              {billions(result['FCFF']):>10,.0f}")
print("=" * 60)

  아모레퍼시픽 CFY2024 연결 재무지표 (억원)

【 손익계산서 】
  매출액               38,851
  영업이익(EBIT)        2,205
  D&A                   2,574
  EBITDA                4,779
  당기순이익             6,016
  지배순이익             5,932

【 NWC 변동 】
  항목                 전기         당기         변동
  ───────────────────────────────────────────
  매출채권            2,929      3,863       +934
  재고자산            3,943      4,978     +1,035
  매입채무              864        960        +95
  ───────────────────────────────────────────
  NWC             6,008      7,882     +1,874
  → ΔNWC = +1,874억  (현금유출↑)

【 CAPEX 】
  유형자산취득             810

【 IBD 】
  단기차입금                3,062
  유동리스부채                 613
  비유동리스부채                728
  ───────────────────────────
  IBD 합계               4,402

【 NOA 】
  [확실한 NOA]  소계: 5,947억
        투자부동산                        5,947
  [기본 NOA]  소계: 8,025억
        현금및현금성자산                     4,515
        단기금융기관예치금                      467
        장기금융기관예치금                      265
        유동 FV

In [51]:
# 셀 45: FCFF 검증 — 영업활동현금흐름 기반 교차검증

# 영업활동현금흐름 추출
cfo_tag = "CashFlowsFromUsedInOperatingActivities"
cfo = get_value(df, cfo_tag, "CF")

print("=== FCFF 계산 방식 비교 ===\n")

# 방식 1: fnguide (영업CF - CAPEX)
capex_val = abs(result['CAPEX'] or 0)
cfo_val   = cfo or 0
fcf_fnguide = cfo_val - capex_val

print(f"【 방식1: fnguide (영업CF - CAPEX) 】")
print(f"  영업활동현금흐름   {cfo_val/1e8:>8,.0f}억")
print(f"  CAPEX            -{capex_val/1e8:>8,.0f}억")
print(f"  FCF              {fcf_fnguide/1e8:>8,.0f}억")

# 방식 2: 우리 FCFF (이론 방식)
ebit      = result['영업이익'] or 0
da        = result['DA'] or 0
delta_nwc = result['delta_nwc'] or 0
tax_rate  = 0.22

nopat    = ebit * (1 - tax_rate)
fcff_our = nopat + da - capex_val - delta_nwc

print(f"\n【 방식2: FCFF 이론 (NOPAT + DA - CAPEX - ΔNWC) 】")
print(f"  NOPAT (EBIT×78%) {nopat/1e8:>8,.0f}억")
print(f"  + D&A            {da/1e8:>8,.0f}억")
print(f"  - CAPEX         -{capex_val/1e8:>8,.0f}억")
print(f"  - ΔNWC          -{delta_nwc/1e8:>8,.0f}억")
print(f"  FCFF             {fcff_our/1e8:>8,.0f}억")

# 방식 3: 영업CF 기반 FCFF (실무 조정)
# 영업CF = NOPAT + DA - ΔNWC + 기타조정 이므로
# FCFF = 영업CF - CAPEX 가 더 정확할 수 있음
print(f"\n【 방식3: 영업CF 기반 (방식1과 동일, 실무 표준) 】")
print(f"  → fnguide와 동일: {fcf_fnguide/1e8:>,.0f}억")

print(f"\n【 차이 원인 분석 】")
print(f"  방식1 vs 방식2 차이: {(fcf_fnguide-fcff_our)/1e8:>+,.0f}억")
print(f"  영업CF({cfo_val/1e8:,.0f}억) vs NOPAT+DA-ΔNWC({(nopat+da-delta_nwc)/1e8:,.0f}억)")
gap = cfo_val - (nopat + da - delta_nwc)
print(f"  갭({gap/1e8:+,.0f}억) = 법인세 실납부액·이자·기타조정 차이")

=== FCFF 계산 방식 비교 ===

【 방식1: fnguide (영업CF - CAPEX) 】
  영업활동현금흐름      3,345억
  CAPEX            -     810억
  FCF                 2,535억

【 방식2: FCFF 이론 (NOPAT + DA - CAPEX - ΔNWC) 】
  NOPAT (EBIT×78%)    1,720억
  + D&A               2,574억
  - CAPEX         -     810억
  - ΔNWC          -   1,874억
  FCFF                1,610억

【 방식3: 영업CF 기반 (방식1과 동일, 실무 표준) 】
  → fnguide와 동일: 2,535억

【 차이 원인 분석 】
  방식1 vs 방식2 차이: +925억
  영업CF(3,345억) vs NOPAT+DA-ΔNWC(2,420억)
  갭(+925억) = 법인세 실납부액·이자·기타조정 차이


In [52]:
# 셀 46: CFO 구성요소 분해 — 갭 원인 추적

print("=== CFO 구성요소 분해 ===\n")

# 주요 CF 조정항목 태그
cf_items = {
    "영업활동현금흐름":      ("CashFlowsFromUsedInOperatingActivities",      "CF"),
    "법인세납부액":          ("IncomeTaxesPaidRefundClassifiedAsOperatingActivities", "CF"),
    "이자수취(영업)":        ("InterestReceivedClassifiedAsOperatingActivities","CF"),
    "이자지급(영업)":        ("InterestPaidClassifiedAsOperatingActivities",   "CF"),
    "배당수취(영업)":        ("DividendsReceivedClassifiedAsOperatingActivities","CF"),
    "영업에서창출된현금":     ("CashFlowsFromUsedInOperations",               "CF"),
}

vals = {}
for kor, (tag, ctx) in cf_items.items():
    v = get_value(df, tag, ctx)
    vals[kor] = v
    print(f"  {kor:20s} {billions(v):>10,.0f}억")

print(f"\n=== FCFF vs CFO 조정 ===\n")

이자비용_세후 = -billions(vals.get("이자지급(영업)") or 0) * (1 - 0.22)
법인세_조정   = billions(vals.get("법인세납부액") or 0)  # 음수
이자수취      = billions(vals.get("이자수취(영업)") or 0)

cfo_억        = billions(cfo)
nopat_da_nwc  = billions(ebit + da - delta_nwc)  # EBIT(1-t)+DA-ΔNWC

print(f"  CFO                      {cfo_억:>8,.0f}억")
print(f"  NOPAT+DA-ΔNWC            {nopat_da_nwc:>8,.0f}억")
print(f"  갭                       {cfo_억-nopat_da_nwc:>+8,.0f}억")
print(f"\n  갭 구성 추정:")
print(f"    이자지급(영업, 세후)    {이자비용_세후:>+8,.0f}억")
print(f"    법인세 납부액 조정      {법인세_조정:>+8,.0f}억")
print(f"    이자수취                {이자수취:>+8,.0f}억")
print(f"    나머지(기타조정)        {cfo_억-nopat_da_nwc-이자비용_세후-법인세_조정-이자수취:>+8,.0f}억")

print(f"\n=== 결론 ===")
print(f"  fnguide FCF(CFO-CAPEX)   {cfo_억-billions(capex_val):>8,.0f}억  ← 실무 스크리닝용")
print(f"  FCFF 이론                {billions(fcff_our):>8,.0f}억  ← DCF 밸류에이션용")
print(f"  두 값은 목적이 달라 차이나는 게 정상")

=== CFO 구성요소 분해 ===

  영업활동현금흐름                  3,345억
  법인세납부액                      873억
  이자수취(영업)                    104억
  이자지급(영업)                    178억
  배당수취(영업)                    478억
  영업에서창출된현금                 3,814억

=== FCFF vs CFO 조정 ===

  CFO                         3,345억
  NOPAT+DA-ΔNWC               2,905억
  갭                           +440억

  갭 구성 추정:
    이자지급(영업, 세후)        -139억
    법인세 납부액 조정          +873억
    이자수취                    +104억
    나머지(기타조정)            -398억

=== 결론 ===
  fnguide FCF(CFO-CAPEX)      2,535억  ← 실무 스크리닝용
  FCFF 이론                   1,610억  ← DCF 밸류에이션용
  두 값은 목적이 달라 차이나는 게 정상


In [54]:
# 셀 48: 한계세율 자동 산출 + FCFF 재계산

def get_marginal_tax_rate(ebit_원):
    """
    한국 법인세 한계세율 (2024년 기준)
    과세표준 구간별 누진세율
    단위: 원
    """
    ebit_억 = ebit_원 / 1e8

    if ebit_억 <= 2:
        rate = 0.110
        구간 = "2억 이하"
    elif ebit_억 <= 200:
        rate = 0.220
        구간 = "2억~200억"
    elif ebit_억 <= 3000:
        rate = 0.242
        구간 = "200억~3000억"
    else:
        rate = 0.275
        구간 = "3000억 초과"

    return rate, 구간

# Terminal Value 기준 EBIT = 당기 EBIT 사용 (정상화 필요시 별도 조정)
ebit_val      = result['영업이익'] or 0
한계세율, 구간 = get_marginal_tax_rate(ebit_val)

da_val    = result['DA'] or 0
capex_val = abs(result['CAPEX'] or 0)
dnwc_val  = result['delta_nwc'] or 0

NOPAT = ebit_val * (1 - 한계세율)
FCFF  = NOPAT + da_val - capex_val - dnwc_val

print("=== DCF용 FCFF (한계세율 적용) ===\n")
print(f"  EBIT              {billions(ebit_val):>8,.0f}억")
print(f"  한계세율 구간      {구간}  →  {한계세율*100:.1f}%")
print(f"  NOPAT             {billions(NOPAT):>8,.0f}억")
print(f"  + D&A           + {billions(da_val):>8,.0f}억")
print(f"  - CAPEX         - {billions(capex_val):>8,.0f}억")
print(f"  - ΔNWC          - {billions(dnwc_val):>8,.0f}억")
print(f"  {'─'*32}")
print(f"  FCFF              {billions(FCFF):>8,.0f}억")

# get_financials 업데이트
print("\n\n=== get_financials() FCFF 수정 ===")
print("  기존: tax_rate = 0.22 고정")
print(f"  수정: get_marginal_tax_rate(EBIT) 자동 적용 → {한계세율*100:.1f}%")

=== DCF용 FCFF (한계세율 적용) ===

  EBIT                 2,205억
  한계세율 구간      200억~3000억  →  24.2%
  NOPAT                1,671억
  + D&A           +    2,574억
  - CAPEX         -      810억
  - ΔNWC          -    1,874억
  ────────────────────────────────
  FCFF                 1,562억


=== get_financials() FCFF 수정 ===
  기존: tax_rate = 0.22 고정
  수정: get_marginal_tax_rate(EBIT) 자동 적용 → 24.2%


In [55]:
# 셀 49: get_financials 함수 FCFF 부분 업데이트

def get_financials(df, year_cur="CFY2024", year_pfy="PFY2023"):

    CTX_IS = f"{year_cur}dFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"
    CTX_BS = f"{year_cur}eFY_ifrs-full_ConsolidatedAndSeparateFinancialStatementsAxis_ifrs-full_ConsolidatedMember"

    def g(tag, ctx="BS", year=year_cur):
        return get_value(df, tag, ctx, year)

    # 손익
    매출액    = g("Revenue",            "IS")
    영업이익   = g("OperatingIncomeLoss", "IS")
    당기순이익  = g("ProfitLoss",          "IS")
    지배순이익  = g("ProfitLossAttributableToOwnersOfParent", "IS")

    DA_통합  = g("DepreciationAndAmortisationExpense", "IS")
    DA_유형  = g("DepreciationExpense",                "IS")
    DA_무형  = g("AmortisationExpense",                "IS")
    DA_사용권 = g("DepreciationRightofuseAssets",       "IS")
    DA = DA_통합 if DA_통합 else sum(x or 0 for x in [DA_유형, DA_무형, DA_사용권])

    # NWC
    def _nwc(year):
        ar  = (g("CurrentTradeReceivables",                      "BS", year) or
               g("ShortTermTradeReceivable",                     "BS", year) or
               g("TradeAndOtherCurrentReceivables",              "BS", year))
        inv = g("Inventories", "BS", year)
        ap  = (g("TradeAndOtherCurrentPayablesToTradeSuppliers", "BS", year) or
               g("ShortTermTradePayables",                       "BS", year) or
               g("TradeAndOtherCurrentPayables",                 "BS", year))
        return {"매출채권": ar, "재고자산": inv, "매입채무": ap,
                "NWC": (ar or 0) + (inv or 0) - (ap or 0)}

    nwc_cur   = _nwc(year_cur)
    nwc_pfy   = _nwc(year_pfy)
    delta_nwc = nwc_cur["NWC"] - nwc_pfy["NWC"]

    # CAPEX
    capex = g("PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities", "CF")

    # IBD
    ibd_tags = {
        "단기차입금":     ("ShorttermBorrowings",               "BS"),
        "유동성장기차입금":("CurrentPortionOfLongtermBorrowings", "BS"),
        "유동리스부채":   ("CurrentLeaseLiabilities",            "BS"),
        "비유동리스부채": ("NoncurrentLeaseLiabilities",         "BS"),
    }
    ibd_사채_유동  = sum(g(t, "BS") or 0 for t in [
        "CurrentPortionOfBonds", "CurrentPortionOfConvertibleBonds",
        "CurrentPortionOfBondWithWarrant", "CurrentPortionOfExchangeableBond"])
    ibd_사채_비유동 = sum(g(t, "BS") or 0 for t in [
        "BondsIssued", "ConvertibleBonds", "BondWithWarrant", "ExchangeableBonds"])

    ibd_detail = {k: g(v[0], v[1]) for k, v in ibd_tags.items()}
    ibd_detail["유동성사채"]  = ibd_사채_유동  or None
    ibd_detail["비유동사채"] = ibd_사채_비유동 or None
    IBD = sum(v or 0 for v in ibd_detail.values())

    # NOA
    noa_items, NOA_억 = calc_noa_v3(df, NOA_RULES_FINAL_V3, CTX_BS, year_cur)

    # FCFF — 한계세율 자동 적용
    ebit_val      = 영업이익 or 0
    한계세율, 구간 = get_marginal_tax_rate(ebit_val)
    NOPAT  = ebit_val * (1 - 한계세율)
    EBITDA = ebit_val + (DA or 0)
    capex_abs = abs(capex or 0)
    FCFF   = NOPAT + (DA or 0) - capex_abs - delta_nwc

    return {
        "year":      year_cur,
        "매출액":    매출액,
        "영업이익":  영업이익,
        "당기순이익": 당기순이익,
        "지배순이익": 지배순이익,
        "DA":        DA,
        "EBITDA":    EBITDA,
        "nwc_cur":   nwc_cur,
        "nwc_pfy":   nwc_pfy,
        "delta_nwc": delta_nwc,
        "CAPEX":     capex,
        "IBD":       IBD,
        "ibd_detail":ibd_detail,
        "NOA":       NOA_억 * 1e8,
        "noa_items": noa_items,
        "한계세율":   한계세율,
        "세율구간":   구간,
        "NOPAT":     NOPAT,
        "FCFF":      FCFF,
    }

# 재실행
result = get_financials(df)

print("=== get_financials() 최종 결과 ===\n")
print(f"  EBIT        {billions(result['영업이익']):>8,.0f}억")
print(f"  세율구간     {result['세율구간']}  ({result['한계세율']*100:.1f}%)")
print(f"  NOPAT       {billions(result['NOPAT']):>8,.0f}억")
print(f"  + D&A     + {billions(result['DA']):>8,.0f}억")
print(f"  - CAPEX   - {abs(billions(result['CAPEX'])):>8,.0f}억")
print(f"  - ΔNWC    - {billions(result['delta_nwc']):>8,.0f}억")
print(f"  {'─'*28}")
print(f"  FCFF        {billions(result['FCFF']):>8,.0f}억")
print(f"\n  NOA         {billions(result['NOA']):>8,.0f}억")
print(f"  IBD         {billions(result['IBD']):>8,.0f}억")

=== get_financials() 최종 결과 ===

  EBIT           2,205억
  세율구간     200억~3000억  (24.2%)
  NOPAT          1,671억
  + D&A     +    2,574억
  - CAPEX   -      810억
  - ΔNWC    -    1,874억
  ────────────────────────────
  FCFF           1,562억

  NOA           14,014억
  IBD            4,402억
